# Tests: `fasterai.quantize.quantizer` (source `nbs/quantize/quantizer.ipynb`)

In [ ]:
from fastcore.test import *
import contextlib
import warnings
import torch
import torch.nn as nn
from torch.ao.quantization import get_default_qconfig_mapping, get_default_qat_qconfig_mapping
from torch.ao.quantization.observer import MinMaxObserver, MovingAverageMinMaxObserver
from fasterai.core.precision import QuantSpec, SPEC_ATTR, _resolve_spec
from fasterai.quantize.quantizer import *
from fasterai.quantize.quantizer import (_ADD_TARGETS, _GRAMMAR_ARGS, _HAS_FQN_CONFIG, _HAS_INT4, _HAS_PT2E,
                                         _HAS_TORCHAO, _LEGACY_QUANT_TYPES, _PlacementQuantizer,
                                         _TORCHAO_CONFIGS, _TorchQuantizer, _batch_input, _conv_add_edges,
                                         _first_input, _is_quantized_weight, _model_device, _node_annotation,
                                         _prepare_pt2e, _pt2e_quantizer, _symmetric_pt2e_config)

if _HAS_PT2E:
    from torch.ao.quantization.fake_quantize import FusedMovingAvgObsFakeQuantize
    from torch.ao.quantization.observer import (HistogramObserver, MovingAveragePerChannelMinMaxObserver,
                                                PerChannelMinMaxObserver)
    from torch.ao.quantization.quantize_pt2e import prepare_pt2e
    from torch.ao.quantization.quantizer import SharedQuantizationSpec
    from torch.ao.quantization.quantizer.xnnpack_quantizer import XNNPACKQuantizer

In [ ]:
from fastcore.test import *

# Construction succeeds with valid backends
q = Quantizer(backend='x86', method='static')
test_eq(q.backend, 'x86')
test_eq(q.method, 'static')

# Different backends
q2 = Quantizer(backend='qnnpack')
test_eq(q2.backend, 'qnnpack')

# Different methods
q3 = Quantizer(method='dynamic')
test_eq(q3.method, 'dynamic')

q4 = Quantizer(method='qat')
test_eq(q4.method, 'qat')

# verbose flag
q5 = Quantizer(verbose=True)
test_eq(q5.verbose, True)

# use_per_tensor flag
q6 = Quantizer(use_per_tensor=True)
test_eq(q6.use_per_tensor, True)

# qconfig_mapping is set by default
assert q.qconfig_mapping is not None

# --- torchao backend ---
if _HAS_TORCHAO:
    # Construction with torchao backend
    qt = Quantizer(backend='torchao', method='int8_weight_only')
    test_eq(qt.backend, 'torchao')
    test_eq(qt.method, 'int8_weight_only')

    # Invalid torchao method raises ValueError
    with ExceptionExpected(ValueError):
        Quantizer(backend='torchao', method='invalid')

    # INT8 weight-only on Linear model
    _m = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 10))
    _mq = Quantizer(backend='torchao', method='int8_weight_only').quantize(_m)
    _out = _mq(torch.randn(2, 64))
    test_eq(_out.shape, (2, 10))
    assert torch.isfinite(_out).all()

    # Original model unchanged (deepcopy)
    test_ne(id(_m), id(_mq))

    # No calibration_dl needed
    _mq2 = Quantizer(backend='torchao', method='int8_weight_only').quantize(_m)
    assert torch.isfinite(_mq2(torch.randn(1, 64))).all()

In [ ]:
# --- pt2e backend: symmetric INT8 config ---
if _HAS_PT2E:
    _cfg = _symmetric_pt2e_config()
    # Activations AND weights are symmetric, so every zero-point is 0
    test_eq(_cfg.input_activation.qscheme, torch.per_tensor_symmetric)
    test_eq(_cfg.output_activation.qscheme, torch.per_tensor_symmetric)
    test_eq((_cfg.input_activation.quant_min, _cfg.input_activation.quant_max), (-127, 127))
    test_eq(_cfg.input_activation.dtype, torch.int8)
    test_eq(_cfg.input_activation.is_dynamic, False)
    # Weights are per-channel over the output-channel axis
    test_eq(_cfg.weight.qscheme, torch.per_channel_symmetric)
    test_eq(_cfg.weight.ch_axis, 0)
    test_eq((_cfg.weight.quant_min, _cfg.weight.quant_max), (-127, 127))
    test_eq(_cfg.bias, None)
    test_eq(_cfg.is_qat, False)
    # ...and post-training observes with plain observers: they record a range, they do not round.
    # The activation observer is pinned to a CLIPPING one on purpose: on a symmetric grid the range
    # the observer reports IS the resolution, and a non-clipping min/max observer lets one heavy
    # calibration tail set the step for the whole tensor.
    _act_ctr = _cfg.input_activation.observer_or_fake_quant_ctr
    test_eq(_act_ctr.p.func, HistogramObserver)
    test_eq(_act_ctr.p.keywords, {'eps': 2 ** -12})
    # The weights keep the observers they had: only the activation grid was ever at stake here
    test_eq(_cfg.weight.observer_or_fake_quant_ctr, PerChannelMinMaxObserver)
    test_eq(_symmetric_pt2e_config(per_channel=False).weight.observer_or_fake_quant_ctr, MinMaxObserver)

    # The QAT arm asks for the very same precision, through fake-quantize modules instead: a training
    # loop has to SEE the rounding error it is correcting, which an observer alone never produces.
    _qat_cfg = _symmetric_pt2e_config(is_qat=True)
    test_eq(_qat_cfg.is_qat, True)
    test_eq(_qat_cfg.input_activation.qscheme, torch.per_tensor_symmetric)
    test_eq(_qat_cfg.weight.qscheme, torch.per_channel_symmetric)
    test_eq(_qat_cfg.weight.ch_axis, 0)
    test_eq((_qat_cfg.weight.quant_min, _qat_cfg.weight.quant_max), (-127, 127))
    for _tensor in (_qat_cfg.input_activation, _qat_cfg.output_activation, _qat_cfg.weight):
        test_eq(_tensor.observer_or_fake_quant_ctr.p.func, FusedMovingAvgObsFakeQuantize)
    test_eq(_qat_cfg.weight.observer_or_fake_quant_ctr.p.keywords['observer'],
            MovingAveragePerChannelMinMaxObserver)
    test_eq(_symmetric_pt2e_config(False, True).weight.observer_or_fake_quant_ctr.p.keywords['observer'],
            MovingAverageMinMaxObserver)  # per-tensor weights watch one range for the whole tensor

    # Construction
    _q = Quantizer(backend='pt2e')
    test_eq((_q.backend, _q.method), ('pt2e', 'static'))

# --- calibration-data helpers ---
# `_batch_input` unwraps (input, target) batches and leaves everything else alone
_x = torch.randn(2, 3)
test_eq(_batch_input((_x, torch.zeros(2))), _x)            # (input, target) batches are unwrapped
test_eq(_batch_input(_x), _x)                              # a bare tensor is passed through
assert _batch_input(_x).data_ptr() == _x.data_ptr()        # `.data` shares storage, it does not copy
_multi = _batch_input([[_x, _x], torch.zeros(2)])   # multi-input batches stay a list
test_eq(len(_multi), 2)
assert _multi[0] is _x

# `_first_input` is the strict one: torch.export needs a real tensor
with ExceptionExpected(TypeError, regex="tensors"):
    _first_input(["not a tensor"])
# ...and it keeps the device the data came on (no forced .cpu())
test_eq(_first_input([torch.randn(2, 3, device='meta')]).device, torch.device('meta'))

# `_model_device` decides where the capture happens
test_eq(_model_device(nn.Linear(4, 4)), torch.device('cpu'))
test_eq(_model_device(nn.Linear(4, 4).to('meta')), torch.device('meta'))
test_eq(_model_device(nn.Module()), torch.device('cpu'))   # no parameters and no buffers
_buffers_only = nn.Module()
_buffers_only.register_buffer('running', torch.zeros(1, device='meta'))
test_eq(_model_device(_buffers_only), torch.device('meta'))

In [ ]:
# --- pt2e backend: end-to-end quantization of a small conv net ---
class _TinyConvNet(nn.Module):
    "Conv-BN-ReLU-Conv-Pool-Linear network, small enough to quantize in a test"
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(3, 8, 3, padding=1), nn.BatchNorm2d(8), nn.ReLU(),
                                      nn.Conv2d(8, 16, 3, padding=1), nn.AdaptiveAvgPool2d(1))
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(16, n_classes))
    def forward(self, x): return self.head(self.features(x))
    def spread_head(self, batch):
        "Center the classifier on the mean feature of `batch`, so its argmax follows the input"
        # An untrained head answers the same class to every input: its bias swamps the differences
        # between feature vectors. Reading the logits from the feature DEVIATIONS is what spreads a
        # batch over classes; the weights stay exactly as initialized, since rescaling them moves
        # every logit by the same factor and leaves the argmax where it was.
        with torch.no_grad():
            self.head[1].bias.copy_(-(self.head[1].weight @ self.features(batch).flatten(1).mean(0)))
        return self

torch.manual_seed(0)
_model = _TinyConvNet().eval()
_calib = [(torch.randn(4, 3, 16, 16), torch.randint(0, 10, (4,))) for _ in range(4)]
_sample = torch.randn(4, 3, 16, 16)

if _HAS_PT2E:
    _qmodel = Quantizer(backend='pt2e').quantize(_model, _calib)
    _out = _qmodel(_sample)
    test_eq(_out.shape, (4, 10))
    assert torch.isfinite(_out).all(), "pt2e model produced non-finite outputs"

    _q_nodes = [n for n in _qmodel.graph.nodes
                if n.op == 'call_function' and 'quantize_per' in str(n.target)]
    # Weights are quantized per channel...
    assert sum('per_channel' in str(n.target) for n in _q_nodes) > 0, "no per-channel weight quantization"
    # ...and every zero-point is 0 (per-tensor ones are literals, per-channel ones are buffers)
    _per_tensor_zps = {n.args[2] for n in _q_nodes if 'per_tensor' in str(n.target)}
    assert _per_tensor_zps, "no per-tensor activation quantization"
    test_eq(_per_tensor_zps, {0})
    assert all(int(b.abs().max()) == 0 for name, b in _qmodel.named_buffers() if 'zero_point' in name)

    # The source model is untouched: still eager modules, still runnable
    test_eq(type(_model.head[1]), nn.Linear)
    assert torch.isfinite(_model(_sample)).all()

    # A fastai-style dataloader (`.one_batch()`) is accepted too
    class _FakeDL(list):
        def one_batch(self): return self[0]
    assert torch.isfinite(Quantizer(backend='pt2e').quantize(_TinyConvNet(), _FakeDL(_calib))(_sample)).all()

    # `max_calibration_samples=None` calibrates over the whole dataloader (verbose path included)
    _all_batches = Quantizer(backend='pt2e', verbose=True).quantize(_TinyConvNet().eval(), _calib,
                                                                   max_calibration_samples=None)
    assert torch.isfinite(_all_batches(_sample)).all()

In [ ]:
# --- pt2e backend: failure modes are loud, never silent ---
if _HAS_PT2E:
    # Missing calibration data
    with ExceptionExpected(ValueError, regex="calibration"):
        Quantizer(backend='pt2e').quantize(_TinyConvNet(), None)

    # Unsupported method: the message says which one IS supported
    with ExceptionExpected(ValueError, regex="static"):
        Quantizer(backend='pt2e', method='dynamic').quantize(_TinyConvNet(), _calib)

    # Calibration data that does not yield tensors
    with ExceptionExpected(TypeError, regex="tensors"):
        Quantizer(backend='pt2e').quantize(_TinyConvNet(), ["not a tensor"])

    # A model torch.export cannot capture must raise — never come back unquantized
    class _DataDependent(nn.Module):
        "Its control flow depends on the data, so torch.export cannot capture it"
        def __init__(self):
            super().__init__()
            self.fc = nn.Linear(4, 4)
        def forward(self, x):
            if x.sum() > 0: return self.fc(x) * 2
            return self.fc(x)

    _dd = _DataDependent().eval()
    try:
        Quantizer(backend='pt2e').quantize(_dd, [torch.randn(2, 4)])
        raise AssertionError("an uncapturable model must raise instead of returning silently")
    except RuntimeError as e:
        assert "torch.export" in str(e), f"unhelpful message: {e}"
        assert e.__cause__ is not None, "the original diagnostic must be chained"
    # The model handed in is untouched and still usable
    test_eq(type(_dd), _DataDependent)
    assert torch.isfinite(_dd(torch.randn(2, 4))).all()

In [ ]:
# --- the legacy backends are left exactly as they were ---
_legacy = Quantizer()
test_eq((_legacy.backend, _legacy.method), ('x86', 'static'))
test_eq((_legacy.use_per_tensor, _legacy.verbose, _legacy.custom_configs), (False, False, None))
test_eq(str(_legacy.qconfig_mapping), str(get_default_qconfig_mapping('x86')))

_legacy_qat = Quantizer(backend='x86', method='qat')
test_eq(str(_legacy_qat.qconfig_mapping), str(get_default_qat_qconfig_mapping('x86')))

# `use_per_tensor` still overrides the default mapping
test_ne(str(Quantizer(use_per_tensor=True).qconfig_mapping), str(get_default_qconfig_mapping('x86')))

In [ ]:
# --- the precision grammar: what the defaults resolve to ---
# Left alone, every backend keeps the precision it has always applied
test_eq(Quantizer().spec.as_dict(),
        {'backend': 'x86', 'method': 'static', 'weight_bits': 8, 'act_bits': 8, 'qscheme': 'per_channel',
         'symmetric': False, 'group_size': None, 'layer_bits': None, 'qdq_placement': None})
test_eq(Quantizer(backend='qnnpack').spec.qscheme, 'per_tensor')
test_eq(Quantizer(method='qat').spec.method, 'qat')       # `method` is the schedule axis, untouched

# The grammar is pure python, so the pt2e cell resolves the same way on a runner whose torch build has
# no pt2e kernels: these assertions run everywhere, and the `Quantizer` ones below run where it does.
test_eq(_resolve_spec('pt2e').qscheme, 'per_channel')
test_eq(_resolve_spec('pt2e').symmetric, True)
# Asking for the default explicitly is the same request: the grammar is a faithful superset
test_eq(_resolve_spec('pt2e', weight_bits=8, act_bits=8, qscheme='per_channel', symmetric=True),
        _resolve_spec('pt2e'))

if _HAS_PT2E:
    # `Quantizer` hands the backend exactly the spec the grammar resolved
    test_eq(Quantizer(backend='pt2e').spec, _resolve_spec('pt2e'))
    test_eq(Quantizer(backend='pt2e', weight_bits=8, act_bits=8, qscheme='per_channel',
                      symmetric=True).spec, Quantizer(backend='pt2e').spec)
else:
    # ...and without those kernels, asking for the backend fails loudly, naming what is missing
    with ExceptionExpected(ImportError, regex="quantize_pt2e"): Quantizer(backend='pt2e')
    with ExceptionExpected(ImportError, regex="quantize_pt2e"):
        Quantizer(backend='pt2e', weight_bits=8, act_bits=8, qscheme='per_channel', symmetric=True)

# `qscheme='per_tensor'` is the grammar's name for `use_per_tensor=True`, and builds the same mapping
test_eq(str(Quantizer(qscheme='per_tensor').qconfig_mapping), str(Quantizer(use_per_tensor=True).qconfig_mapping))
test_eq(Quantizer(qscheme='per_tensor').use_per_tensor, True)
# ...while a backend whose default already IS per-tensor does not go down that path
test_eq(Quantizer(backend='qnnpack').use_per_tensor, False)
test_eq(str(Quantizer(backend='qnnpack').qconfig_mapping), str(get_default_qconfig_mapping('qnnpack')))

# On torchao, naming a precision names the recipe that applies it (and vice versa)
if _HAS_TORCHAO:
    test_eq(Quantizer(backend='torchao', method='int8_weight_only').spec.label, 'W8A16')
    test_eq(Quantizer(backend='torchao', method='int8_dynamic').spec.label, 'W8A8')
    test_eq(Quantizer(backend='torchao', weight_bits=8, act_bits=16).method, 'int8_weight_only')
    test_eq(Quantizer(backend='torchao', weight_bits=8, act_bits=8).method, 'int8_dynamic')
    # a group size reaches the torchao configuration instead of being dropped on the floor
    _grouped = Quantizer(backend='torchao', weight_bits=8, act_bits=16, qscheme='per_group', group_size=32)
    test_eq(_grouped.spec.group_size, 32)
    test_eq(_grouped._torchao_config().group_size, 32)
    test_eq(Quantizer(backend='torchao', method='int8_weight_only')._torchao_config().group_size, None)

In [ ]:
# --- a precision a backend cannot honor is refused AT CONSTRUCTION, never approximated ---
def _refused(regex, exc=ValueError, **kwargs):
    "Assert `Quantizer(**kwargs)` refuses to be built, with a message matching `regex`"
    with ExceptionExpected(exc, regex=regex): Quantizer(**kwargs)

# The pt2e requests below need no `_HAS_PT2E` guard: the grammar is resolved before the backend is
# looked for, so they raise the same ValueError on a runner whose torch build has no pt2e kernels.

# a precision no backend runs
_refused("No fasterai backend runs W4A8", backend='pt2e', weight_bits=4, act_bits=8)
_refused("W4A8", backend='torchao', weight_bits=4, act_bits=8)
# a precision another backend runs
_refused("that run W8A16", backend='pt2e', act_bits=16)
# affine observers cannot be asked to be symmetric — the exact silent drop this grammar exists to stop
_refused("cannot honor symmetric=True", backend='x86', symmetric=True)
_refused("cannot honor symmetric=True", backend='fbgemm', symmetric=True)
# per-group is a torchao axis, and needs a size
_refused("quantize per group", backend='x86', group_size=64)
_refused("that do: ", backend='pt2e', qscheme='per_group', group_size=64)
# per-layer widths need a backend that can configure one module at a time
_refused("per-layer `weight_bits` dict", backend='pt2e', weight_bits={'head.1': 16})
# torchao configures one Linear at a time weight-only, and ONLY there: its W8A8 recipe recomputes the
# activation scales at run time. The refusal names the argument that reaches the cell that can.
_refused("only at W8A16: add act_bits=16", backend='torchao', weight_bits={'fc': 8})
_refused("only at W8A16: add act_bits=16", backend='torchao', method='int8_dynamic',
         weight_bits={'fc': 8})
_refused("only at W8A16: add act_bits=16", backend='torchao', weight_bits={'fc': 16}, act_bits=8)
# two ways of asking for opposite things
_refused("different weight axes", use_per_tensor=True, qscheme='per_channel')
_refused("never read", backend='pt2e', use_per_tensor=True)
# a hand-written mapping and the grammar cannot both describe the configuration
_refused("Keep one of the two", qconfig_mapping=get_default_qconfig_mapping('x86'), symmetric=False)

# --- plausible-wrong values, from the user's side ---
_refused("must be an int", exc=TypeError, backend='pt2e', weight_bits='8')
_refused("Unknown qscheme 'channel'", backend='pt2e', qscheme='channel')
_refused("is not a width", backend='pt2e', act_bits=32)
_refused("True, False or None", exc=TypeError, backend='pt2e', symmetric='yes')
_refused("Unknown backend 'x87'", backend='x87')

# ...and a legacy mapping handed in alone still works, untouched
test_eq(str(Quantizer(qconfig_mapping=get_default_qconfig_mapping('x86')).qconfig_mapping),
        str(get_default_qconfig_mapping('x86')))

if _HAS_TORCHAO:
    # ...and asked at W8A16 it is not refused at all: the dict reaches the backend
    test_eq(Quantizer(backend='torchao', weight_bits={'fc': 16}, act_bits=16).spec.layer_bits, {'fc': 16})
    test_eq(Quantizer(backend='torchao', method='int8_weight_only', weight_bits={'fc': 16}).method,
            'int8_weight_only')
    # a recipe and a precision that disagree
    _refused("contradicts", backend='torchao', method='int8_weight_only', act_bits=8)
    # an unknown recipe, with or without kernels installed
    _refused("has no method 'invalid'", backend='torchao', method='invalid')
    # INT4 is a torchao-only, group-wise cell: it resolves, then needs the kernels to actually run
    _int4 = _resolve_spec('torchao', weight_bits=4)
    test_eq((_int4.label, _int4.qscheme, _int4.group_size), ('W4A16', 'per_group', 128))
    if not _HAS_INT4:
        _refused("not available in this environment", backend='torchao', weight_bits=4)

In [ ]:
# --- pt2e: the grammar changes the artifact, and the artifact matches the spec ---
def _n_per_channel(graph_module):
    "Number of per-channel quantize/dequantize nodes in a converted pt2e graph"
    return sum('per_channel' in str(n.target) for n in graph_module.graph.nodes
               if n.op == 'call_function' and 'quantize_per' in str(n.target))

def _same_state(a, b):
    "Whether two models hold exactly the same state_dict — same keys, same values"
    sa, sb = a.state_dict(), b.state_dict()
    return list(sa) == list(sb) and all(torch.equal(sa[k], sb[k]) for k in sa)

if _HAS_PT2E:
    # asking for the default explicitly must produce the very same artifact
    _explicit = Quantizer(backend='pt2e', weight_bits=8, act_bits=8, qscheme='per_channel',
                          symmetric=True).quantize(_model, _calib)
    assert _same_state(_explicit, _qmodel), "the explicit grammar changed the default artifact"
    test_eq(_n_per_channel(_explicit), _n_per_channel(_qmodel))
    assert _n_per_channel(_explicit) > 0

    # per-tensor weights: fewer scales, and the per-channel nodes are gone
    _pt = Quantizer(backend='pt2e', qscheme='per_tensor').quantize(_model, _calib)
    test_eq(_n_per_channel(_pt), 0)
    assert torch.isfinite(_pt(_sample)).all()
    assert not _same_state(_pt, _qmodel), "per-tensor and per-channel produced the same artifact"

    # symmetric=False buys range at the cost of portability, and says so
    with warnings.catch_warnings(record=True) as _caught:
        warnings.simplefilter('always')
        _affine_q = Quantizer(backend='pt2e', symmetric=False)
    assert any('zero-point' in str(w.message) for w in _caught), [str(w.message) for w in _caught]
    test_eq(_pt2e_quantizer(_affine_q.spec).global_config.input_activation.qscheme, torch.per_tensor_affine)
    test_eq(_pt2e_quantizer(Quantizer(backend='pt2e').spec).global_config.input_activation.qscheme,
            torch.per_tensor_symmetric)
    _affine = _affine_q.quantize(_model, _calib)
    assert torch.isfinite(_affine(_sample)).all()

    # --- provenance: the spec travels with the model it produced ---
    from fasterai.core.precision import quant_spec
    _q = Quantizer(backend='pt2e')
    _tagged = _q.quantize(_model, _calib)
    assert quant_spec(_tagged) is _q.spec
    test_eq(quant_spec(_tagged).as_dict()['qscheme'], 'per_channel')
    test_eq(quant_spec(_pt).qscheme, 'per_tensor')            # the axis that actually ran
    test_eq(quant_spec(_tagged).exports, True)
    with ExceptionExpected(AttributeError):                   # frozen: provenance cannot be rewritten
        quant_spec(_tagged).qscheme = 'per_tensor'
    # the source model carries no provenance: it was not quantized
    test_eq(quant_spec(_model), None)

In [ ]:
# --- per-layer widths: the layer named 16 really is left in floating point ---
def _module_kind(model, name):
    "Fully-qualified class of a submodule, which tells a quantized layer from a float one"
    m = model.get_submodule(name)
    return f"{type(m).__module__}.{type(m).__name__}"

def _is_float_module(kind):
    "Whether that class is a float one — FX leaves fused-but-unquantized modules float too"
    return kind.startswith('torch.nn.modules') or 'intrinsic.modules' in kind

_uniform = Quantizer().quantize(_TinyConvNet().eval(), _calib)
assert _uniform is not _model, "legacy static quantization silently returned the source model"
assert 'quantized' in _module_kind(_uniform, 'head.1'), _module_kind(_uniform, 'head.1')

_mixed = Quantizer(weight_bits={'head.1': 16}).quantize(_TinyConvNet().eval(), _calib)
test_eq(_module_kind(_mixed, 'head.1'), 'torch.nn.modules.linear.Linear')   # left in floating point
assert 'quantized' in _module_kind(_mixed, 'features.3'), "the rest of the model must still be INT8"
test_eq(_mixed._fasterai_quant_spec.layer_bits, {'head.1': 16})
test_eq(_mixed._fasterai_quant_spec.weight_bits, 8)   # the layers the dict does not name

# the legacy backends quantize convolutions too, so a dict may name one — and it is honored
_mixed_conv = Quantizer(weight_bits={'features.3': 16}).quantize(_TinyConvNet().eval(), _calib)
test_eq(_module_kind(_mixed_conv, 'features.3'), 'torch.nn.modules.conv.Conv2d')
assert 'quantized' in _module_kind(_mixed_conv, 'head.1'), "the rest of the model must still be INT8"

# ...and so may a module CONTAINING them: the FX flow resolves a module-name qconfig by walking parents,
# so one entry covers a whole subtree
_container = Quantizer(weight_bits={'features': 16}).quantize(_TinyConvNet().eval(), _calib)
assert _is_float_module(_module_kind(_container, 'features.3')), _module_kind(_container, 'features.3')
assert 'quantized' in _module_kind(_container, 'head.1'), "only the named subtree may stay float"
# ...and the accounting agrees with the artifact: a container 16 is a 16 on everything under it
_by_subtree = Quantizer(weight_bits={'features': 16})
test_eq((_by_subtree._layer_width('features.3'), _by_subtree._layer_width('head.1')), (16, 8))

# a layer name the model does not have is a typo, not a silent no-op
with ExceptionExpected(ValueError, regex="does not have"):
    Quantizer(weight_bits={'head.42': 16}).quantize(_TinyConvNet().eval(), _calib)

# ...and a layer this backend does not quantize (features.2 is a ReLU) is refused rather than ignored:
# a width there used to be accepted and do nothing at all
with ExceptionExpected(ValueError, regex="does not quantize"):
    Quantizer(weight_bits={'features.2': 16}).quantize(_TinyConvNet().eval(), _calib)

# an 8 inside a subtree already left at 16 cannot be honored — only the 16 is written into the mapping,
# so the outer entry wins and the 8 would be ignored
with ExceptionExpected(ValueError, regex="inside a module it also leaves at 16"):
    Quantizer(weight_bits={'features': 16, 'features.3': 8}).quantize(_TinyConvNet().eval(), _calib)

# a dict leaving EVERY quantizable layer at 16 used to return a model with not one quantized layer,
# carrying a quantized model's provenance — by leaf name...
with ExceptionExpected(ValueError, regex="leaves all"):
    Quantizer(weight_bits={'features.0': 16, 'features.1': 16, 'features.3': 16, 'head.1': 16}).quantize(
        _TinyConvNet().eval(), _calib)
# ...and through the containers that blanket them
with ExceptionExpected(ValueError, regex="leaves all"):
    Quantizer(weight_bits={'features': 16, 'head': 16}).quantize(_TinyConvNet().eval(), _calib)

In [ ]:
# --- which module types a per-layer width can be HONORED on: measured against torch itself ---
# The accept-list is read from torch's own default static mapping rather than written by hand. These
# are the measurements that justify it: for each type, "does the default x86 flow quantize it?" and
# "does naming it 16 leave it in floating point?" must agree with whether a width on it is accepted.
def _honors(layer, sample, wrap=None):
    "Quantize a one-layer model unnamed then named 16, and report (was quantized, the 16 was honored)"
    class _Net(nn.Module):
        def __init__(self):
            super().__init__(); self.target = layer(); self.fc = nn.Linear(4, 4)
        def forward(self, x): return self.fc(wrap(self.target(x)) if wrap else self.target(x))
    _cal = [(sample, torch.zeros(sample.shape[0], dtype=torch.long)) for _ in range(2)]
    torch.manual_seed(0)
    _plain = _module_kind(Quantizer().quantize(_Net().eval(), _cal), 'target')
    torch.manual_seed(0)
    _named = _module_kind(Quantizer(weight_bits={'target': 16, 'fc': 8}).quantize(_Net().eval(), _cal),
                          'target')
    return not _is_float_module(_plain), _is_float_module(_named)

# accepted, because torch quantizes them AND a 16 leaves them float — including the ones a hand-written
# "convolutions and Linear" list would have refused
test_eq(_honors(lambda: nn.Conv2d(3, 4, 3, padding=1), torch.randn(2, 3, 4, 4), lambda t: t.mean((-1, -2))),
        (True, True))
test_eq(_honors(lambda: nn.ConvTranspose2d(3, 4, 3, padding=1), torch.randn(2, 3, 4, 4),
                lambda t: t.mean((-1, -2))), (True, True))
test_eq(_honors(lambda: nn.Linear(4, 4), torch.randn(2, 4)), (True, True))
test_eq(_honors(lambda: nn.LayerNorm(4), torch.randn(2, 4)), (True, True))

# refused, because the default flow never rewrites them: a width there is a request that cannot land
class _EmbNet(nn.Module):
    "nn.Embedding IS in torch's static mapping, but quantizing it needs an observer this flow never sets"
    def __init__(self):
        super().__init__(); self.emb = nn.Embedding(20, 8); self.fc = nn.Linear(8, 4)
    def forward(self, x): return self.fc(self.emb(x).mean(1))

_emb_cal = [(torch.randint(0, 20, (4, 5)), torch.zeros(4, dtype=torch.long)) for _ in range(2)]
_emb_q = Quantizer().quantize(_EmbNet().eval(), _emb_cal)
assert _is_float_module(_module_kind(_emb_q, 'emb')), "the default flow is expected to skip Embedding"
assert 'quantized' in _module_kind(_emb_q, 'fc')
with ExceptionExpected(ValueError, regex="does not quantize"):
    Quantizer(weight_bits={'emb': 16, 'fc': 8}).quantize(_EmbNet().eval(), _emb_cal)
# ...including the shape that used to hand back a model with not one quantized layer, plus provenance
with ExceptionExpected(ValueError, regex="does not quantize"):
    Quantizer(weight_bits={'emb': 8, 'fc': 16}).quantize(_EmbNet().eval(), _emb_cal)

class _RnnNet(nn.Module):
    "nn.LSTM is not in the static mapping at all: it belongs to the dynamic flow"
    def __init__(self):
        super().__init__(); self.rnn = nn.LSTM(8, 8, batch_first=True); self.fc = nn.Linear(8, 4)
    def forward(self, x): return self.fc(self.rnn(x)[0].mean(1))

_rnn_cal = [(torch.randn(4, 5, 8), torch.zeros(4, dtype=torch.long)) for _ in range(2)]
assert _is_float_module(_module_kind(Quantizer().quantize(_RnnNet().eval(), _rnn_cal), 'rnn'))
with ExceptionExpected(ValueError, regex="does not quantize"):
    Quantizer(weight_bits={'rnn': 16, 'fc': 8}).quantize(_RnnNet().eval(), _rnn_cal)

# the accept-list is torch's mapping, minus the two types its default qconfig cannot observe
assert {nn.Conv2d, nn.ConvTranspose2d, nn.Linear, nn.LayerNorm} <= set(_LEGACY_QUANT_TYPES)
for _t in (nn.Embedding, nn.EmbeddingBag, nn.LSTM, nn.GRU, nn.RNN, nn.ReLU):
    assert _t not in _LEGACY_QUANT_TYPES, _t

# --- method='dynamic' reads no per-module configuration, so a dict is refused, never ignored ---
# `quantize_dynamic` rewrites every Linear/RNN off a type list: the dict used to be accepted, dropped,
# and then recorded on the model as if it had been applied.
with ExceptionExpected(ValueError, regex="method='dynamic'"):
    Quantizer(backend='x86', method='dynamic', weight_bits={'fc': 16})
with ExceptionExpected(ValueError, regex="'static', 'qat'"):
    Quantizer(backend='qnnpack', method='dynamic', weight_bits={'fc': 8})
# ...while dynamic without a dict, and the methods that can honor one, are untouched
test_eq(Quantizer(backend='x86', method='dynamic').spec.layer_bits, None)
test_eq(Quantizer(backend='x86', method='qat', weight_bits={'fc': 16}).spec.layer_bits, {'fc': 16})

In [ ]:
from fasterai.core.precision import quant_spec

# --- provenance is never a fiction: a spec is recorded only when it describes what ran ---
_mapping = get_default_qconfig_mapping('x86')

# a caller-supplied mapping decides the configuration, so there is no spec to record
_byo = Quantizer(qconfig_mapping=_mapping).quantize(_TinyConvNet().eval(), _calib)
assert 'quantized' in _module_kind(_byo, 'features.3'), "the model was not quantized at all"
test_eq(quant_spec(_byo), None)
# ...and mixing that mapping with any precision argument, grammar or legacy flag, is refused
with ExceptionExpected(ValueError, regex="Keep one of the two"):
    Quantizer(qconfig_mapping=_mapping, use_per_tensor=True)
with ExceptionExpected(ValueError, regex="Keep one of the two"):
    Quantizer(qconfig_mapping=_mapping, qscheme='per_tensor')

# the default path still records one
test_eq(quant_spec(Quantizer().quantize(_TinyConvNet().eval(), _calib)).backend, 'x86')

# --- torchao rewrites Linear layers only: a model it cannot touch must raise, not come back "quantized" ---
if _HAS_TORCHAO:
    _conv_only = nn.Sequential(nn.Conv2d(3, 8, 3), nn.ReLU(), nn.AdaptiveAvgPool2d(1)).eval()
    with ExceptionExpected(ValueError, regex="rewrites Linear layers only"):
        Quantizer(backend='torchao', method='int8_weight_only').quantize(_conv_only)
    # a model with Linear layers still goes through
    assert quant_spec(Quantizer(backend='torchao', method='int8_weight_only').quantize(
        nn.Sequential(nn.Linear(8, 8)).eval())).label == 'W8A16'

    # the recipes this environment can run are a subset of the ones the grammar knows
    from fasterai.core.precision import _TORCHAO_CELL
    assert set(_TORCHAO_CONFIGS) <= set(_TORCHAO_CELL), (set(_TORCHAO_CONFIGS), set(_TORCHAO_CELL))

In [ ]:
# --- per-layer widths on torchao: the same quantization as the uniform path, on the layers named 8 ---
import hashlib
import io

def _state_bytes(model):
    "Serialized state_dict — the artifact two quantization paths have to agree on, byte for byte"
    buf = io.BytesIO(); torch.save(model.state_dict(), buf); return buf.getvalue()

def _state_digest(model):
    "SHA-256 of those bytes: the same claim as comparing them, in a form a failure can print"
    return hashlib.sha256(_state_bytes(model)).hexdigest()

def _mlp():
    "Two Linear layers, deterministically initialized, so two runs are comparable byte for byte"
    torch.manual_seed(0)
    return nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10)).eval()

def _weight_root(model, name):
    "Package the class of a layer's weight comes from: 'torch' when float, 'torchao' when quantized"
    return type(model.get_submodule(name).weight).__module__.split('.')[0]

test_eq(_state_digest(_mlp()), _state_digest(_mlp()))   # the fixture itself is deterministic

if _HAS_TORCHAO and _HAS_FQN_CONFIG:
    # GOLDEN: a dict naming every Linear 8 must produce EXACTLY the uniform artifact. The per-layer path
    # selects and configures the layers itself, so any drift in either shows up here as different bytes.
    _uniform_ao = Quantizer(backend='torchao', method='int8_weight_only').quantize(_mlp())
    _all_eight = Quantizer(backend='torchao', weight_bits={'0': 8, '2': 8}, act_bits=16).quantize(_mlp())
    test_eq(_state_digest(_all_eight), _state_digest(_uniform_ao))
    test_eq(quant_spec(_all_eight).layer_bits, {'0': 8, '2': 8})

    # ARTIFACT: read off the model, not off the request — the layer named 16 keeps a torch weight, the
    # layer the dict does not name is quantized at the uniform width
    _mixed_ao = Quantizer(backend='torchao', weight_bits={'0': 16}, act_bits=16).quantize(_mlp())
    test_eq((_weight_root(_mixed_ao, '0'), _weight_root(_mixed_ao, '2')), ('torch', 'torchao'))
    test_eq((_is_quantized_weight(_mixed_ao[0]), _is_quantized_weight(_mixed_ao[2])), (False, True))
    _mixed_out = _mixed_ao(torch.randn(4, 64))
    test_eq(_mixed_out.shape, (4, 10))
    assert torch.isfinite(_mixed_out).all(), "a per-layer torchao model produced non-finite outputs"
    # ...and it is a real mixture: neither of the two artifacts it sits between
    assert _state_digest(_mixed_ao) != _state_digest(_uniform_ao), "nothing was left in floating point"
    assert _state_digest(_mixed_ao) != _state_digest(_mlp()), "nothing was quantized"

    # PROVENANCE: the spec pins the per-layer request AND the width the layers it does not name got
    _spec_ao = quant_spec(_mixed_ao)
    test_eq((_spec_ao.layer_bits, _spec_ao.weight_bits, _spec_ao.act_bits), ({'0': 16}, 8, 16))
    test_eq((_spec_ao.backend, _spec_ao.method, _spec_ao.label), ('torchao', 'int8_weight_only', 'W8A16'))
    test_eq(_spec_ao.exports, False)     # weight-only stays unexportable, per-layer or not
    test_eq(quant_spec(_mlp()), None)    # the model handed in is untouched

    # the spec holds a copy of the caller's dict, so mutating it afterwards cannot rewrite provenance
    _asked = {'0': 16}
    _copied = Quantizer(backend='torchao', weight_bits=_asked, act_bits=16)
    _asked['0'] = 8
    test_eq(_copied.spec.layer_bits, {'0': 16})
elif _HAS_TORCHAO:
    # torchao installed, but too old to expose FqnToConfig: refused at construction, naming the fix
    with ExceptionExpected(ImportError, regex="FqnToConfig"):
        Quantizer(backend='torchao', weight_bits={'0': 16}, act_bits=16)
else:
    # no torchao at all (the CI runner): the request is refused at construction, naming what to install
    with ExceptionExpected(ImportError, regex="pip install torchao"):
        Quantizer(backend='torchao', weight_bits={'0': 16}, act_bits=16)

In [ ]:
# --- per-layer widths on torchao: the names a dict may not carry, and what verbose reports ---
if _HAS_TORCHAO and _HAS_FQN_CONFIG:
    def _refused_on(regex, model, exc=ValueError, **kwargs):
        "Assert quantizing `model` with these arguments is refused, with a message matching `regex`"
        with ExceptionExpected(exc, regex=regex): Quantizer(**kwargs).quantize(model)

    _convnet = _TinyConvNet().eval()   # convolutions plus a single Linear (head.1)

    # a convolution torchao never rewrites: a width on it would be silently dropped
    _refused_on(r"does not quantize: \['features.3'\]", _convnet,
                backend='torchao', weight_bits={'features.3': 16, 'head.1': 8}, act_bits=16)
    # ...and a name the model does not have at all is still the earlier, different refusal
    _refused_on("does not have", _convnet, backend='torchao', weight_bits={'nope': 16}, act_bits=16)
    # ...while a model torchao cannot touch AT ALL gets the backend answer, not a remark about names
    with ExceptionExpected(ValueError, regex="Use backend='x86'"):
        Quantizer(backend='torchao', weight_bits={'0': 16}, act_bits=16).quantize(
            nn.Sequential(nn.Conv2d(3, 4, 3)).eval())

    # every layer torchao can quantize left at 16 = a model that was never quantized
    _refused_on("leaves all 2 layer", _mlp(),
                backend='torchao', weight_bits={'0': 16, '2': 16}, act_bits=16)
    _refused_on("leaves all 1 layer", _convnet,
                backend='torchao', weight_bits={'head.1': 16}, act_bits=16)

    # `MultiheadAttention.out_proj` is an nn.Linear SUBCLASS that torchao's own filter skips. Both paths
    # must agree about it: naming it is refused, and it stays in floating point either way.
    class _AttnNet(nn.Module):
        "One MultiheadAttention (whose out_proj torchao skips) and one plain Linear"
        def __init__(self):
            super().__init__()
            self.attn = nn.MultiheadAttention(embed_dim=8, num_heads=2, batch_first=True)
            self.fc = nn.Linear(8, 4)
        def forward(self, x): return self.fc(self.attn(x, x, x)[0])

    def _attn_net():
        "A deterministically initialized `_AttnNet`, so two quantizations are comparable"
        torch.manual_seed(0); return _AttnNet().eval()

    # ...so a model whose ONLY Linear is one torchao skips must raise: quantizing it would rewrite
    # nothing at all and still hand back a model carrying a quantized model's provenance
    class _AttnOnly(nn.Module):
        "Its only nn.Linear is MultiheadAttention.out_proj, which torchao never rewrites"
        def __init__(self):
            super().__init__()
            self.attn = nn.MultiheadAttention(embed_dim=8, num_heads=2, batch_first=True)
        def forward(self, x): return self.attn(x, x, x)[0]

    with ExceptionExpected(ValueError, regex="Linear layers torchao skips"):
        Quantizer(backend='torchao', method='int8_weight_only').quantize(_AttnOnly().eval())

    _an = _attn_net()
    assert isinstance(_an.attn.out_proj, nn.Linear)   # it IS an nn.Linear...
    test_eq(Quantizer(backend='torchao', weight_bits={'fc': 8}, act_bits=16)._quantizable_names(_an),
            ['fc'])                                   # ...and torchao still does not rewrite it
    _refused_on(r"does not quantize: \['attn.out_proj'\]", _an,
                backend='torchao', weight_bits={'attn.out_proj': 16, 'fc': 8}, act_bits=16)

    _an_uniform = Quantizer(backend='torchao', method='int8_weight_only').quantize(_attn_net())
    _an_dict = Quantizer(backend='torchao', weight_bits={'fc': 8}, act_bits=16).quantize(_attn_net())
    for _m in (_an_uniform, _an_dict):
        test_eq(_weight_root(_m, 'attn.out_proj'), 'torch')
        test_eq(_weight_root(_m, 'fc'), 'torchao')
        # `in_proj_weight` is a bare Parameter of the attention module, not a layer: never counted
        test_eq(type(_m.attn.in_proj_weight).__module__.split('.')[0], 'torch')
    test_eq(_state_digest(_an_dict), _state_digest(_an_uniform))

    # --- verbose reports what it asked for, then counts what actually came back quantized ---
    _log = io.StringIO()
    with contextlib.redirect_stdout(_log):
        Quantizer(backend='torchao', weight_bits={'0': 16}, act_bits=16, verbose=True).quantize(_mlp())
    _log = _log.getvalue()
    assert "1/2 Linear layers" in _log, _log
    assert "1 left in floating point: 0" in _log, _log
    assert "quantized 1 layers" in _log, _log
    # the post-hoc count reads the weight CLASS: the attribute it used to probe no longer exists, so the
    # uniform path used to report zero quantized layers on a model where it had quantized both
    _log = io.StringIO()
    with contextlib.redirect_stdout(_log):
        Quantizer(backend='torchao', method='int8_weight_only', verbose=True).quantize(_mlp())
    assert "quantized 2 layers" in _log.getvalue(), _log.getvalue()
    assert not any(hasattr(getattr(m, 'weight', None), 'layout_type') for m in _uniform_ao.modules())
else:
    # The CI runner has no torchao: what it CAN run is the grammar, which is pure python — the request
    # resolves, and the near-miss cell still names the argument that reaches the one that honors it.
    with ExceptionExpected(ImportError, regex="torchao"):
        Quantizer(backend='torchao', weight_bits={'fc': 16}, act_bits=16)
    test_eq(_resolve_spec('torchao', weight_bits={'fc': 16}, act_bits=16).layer_bits, {'fc': 16})
    with ExceptionExpected(ValueError, regex="only at W8A16: add act_bits=16"):
        _resolve_spec('torchao', weight_bits={'fc': 8})

In [ ]:
# --- end-to-end: pt2e INT8 → QDQ ONNX (needs onnx/onnxscript/onnxruntime) ---
import numpy as np
import tempfile
from pathlib import Path
from fasterai.export.onnx_exporter import ONNXModel, _has_package, export_qdq, qdq_stats, verify_qdq

# The parity evidence on THIS model is a LOGIT comparison, not an argmax one: an untrained model on
# noise answers the same class to every input, so argmax agreement is 1.0 even against a graph fed
# different data. Measured on this model (torch 2.9.1, onnxruntime CPU): max|Δlogit| = 2.9e-3 between
# the quantized PyTorch model and its export on the SAME batch — ONNX Runtime runs real INT8 kernels,
# so it is not bit-exact — against 2.1e-2 when the ONNX arm is fed a DIFFERENT batch. `_ATOL` sits
# between the two, and the assertion that follows proves it: a wrong-input harness turns this red.
_ATOL = 8e-3

if _HAS_PT2E and all(_has_package(p) for p in ('onnx', 'onnxscript', 'onnxruntime')):
    with tempfile.TemporaryDirectory() as _tmp:
        _onnx_path = export_qdq(_qmodel, _sample, Path(_tmp)/'tiny_qdq.onnx')
        _stats = qdq_stats(_onnx_path)
        assert _stats.n_quantize > 0 and _stats.n_dequantize > 0, _stats
        assert _stats.n_per_channel > 0, "per-channel weights did not survive the export"
        test_eq(_stats.n_nonzero_zero_point, 0)  # portability: zero_point == 0 everywhere

        _session = ONNXModel(_onnx_path)
        with torch.no_grad(): _pt_logits = _qmodel(_sample).numpy()
        _onnx_logits = _session(_sample).numpy()
        assert np.allclose(_pt_logits, _onnx_logits, atol=_ATOL), \
            f"max|Δlogit| = {np.abs(_pt_logits - _onnx_logits).max():.3e}"
        # ...and the pin has power: the same assertion against a different batch must fail
        _other = _session(torch.randn(4, 3, 16, 16)).numpy()
        assert not np.allclose(_pt_logits, _other, atol=_ATOL), \
            "the parity check cannot tell one input batch from another — it proves nothing"

        # --- argmax agreement, read on predictions that actually VARY ---
        # An agreement is evidence only when the reference model does not answer the same class to
        # every input: on an untrained net it reads 1.0 against ANY graph. `spread_head` is what gives
        # the reference predictions that follow the input, and their spread is asserted BEFORE the
        # agreement, so this check cannot go vacuous again without turning red.
        # Measured here (torch 2.9.1, onnxruntime CPU): over the 32 probes the reference spans 8
        # classes, agrees with its OWN exported graph on 32 of 32 (1.0) and with the OTHER model's
        # graph on 3 of 32 (0.094) — the constants sit between the two. 0.9 where this check used to
        # read 0.99 is deliberate: on 32 probes it tolerates at most 3 disagreements, and a reference
        # whose predictions vary sits near decision boundaries, where ONNX Runtime's INT8 kernels
        # round differently than PyTorch's — the same fixture in export/onnx_exporter.ipynb reads
        # 0.938, which 0.99 would call a failure.
        _MIN_CLASSES, _MIN_AGREEMENT, _MAX_FOREIGN_AGREEMENT = 4, 0.9, 0.5

        def _spread_model():
            "A quantized `_TinyConvNet` whose predictions span classes instead of collapsing on one"
            return Quantizer(backend='pt2e').quantize(
                _TinyConvNet().eval().spread_head(torch.randn(64, 3, 16, 16)), _calib)

        # The fork is defense in depth: no cell below reads this stream today (the next one to
        # draw reseeds), and these extra draws must not become what moves a cell added later.
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(0)
            _reference, _foreign = _spread_model(), _spread_model()  # same recipe, different weights
            _spread_probe = torch.randn(32, 3, 16, 16)
        _reference_path = export_qdq(_reference, _sample, Path(_tmp)/'spread_reference.onnx')
        _foreign_path = export_qdq(_foreign, _sample, Path(_tmp)/'spread_foreign.onnx')

        _static_batch = _sample.shape[0]  # the exported graph only accepts the batch it was traced on
        _n_batches = _spread_probe.shape[0] // _static_batch
        with torch.no_grad():
            _preds = torch.cat([_reference(_c).argmax(-1)
                                for _c in _spread_probe.split(_static_batch)]).tolist()
        assert len(set(_preds)) >= _MIN_CLASSES, ("the reference predictions must span classes, or "
                                                  f"the agreement below proves nothing: {_preds}")

        with warnings.catch_warnings(record=True) as _caught:
            warnings.simplefilter('always')
            _agreement = verify_qdq(_reference, _reference_path, _spread_probe, n_batches=_n_batches)
        assert _agreement >= _MIN_AGREEMENT, f"ONNX and PyTorch disagree too often: {_agreement}"
        assert not any('vacuous' in str(w.message) for w in _caught), [str(w.message) for w in _caught]
        # ...and this pin has power too: the same probe read against ANOTHER model's graph collapses
        _foreign_agreement = verify_qdq(_reference, _foreign_path, _spread_probe, n_batches=_n_batches)
        assert _foreign_agreement <= _MAX_FOREIGN_AGREEMENT, \
            f"the agreement cannot tell this graph from another model's: {_foreign_agreement}"

In [ ]:
# --- the produced file is one an ONNX parser can read: `kernel_shape` on Conv, and the opset asked for ---
# (they run here, not in the exporter notebook, which the test suite does not execute; the reason the
#  attribute has to be written at all is documented on `_set_conv_kernel_shape` itself)
from fasterai.export.onnx_exporter import _check_produced_opset, _set_conv_kernel_shape

if _has_package('onnx'):
    import onnx
    from onnx import TensorProto, helper, numpy_helper
    from onnx.external_data_helper import convert_model_to_external_data

    # how the weight reaches the convolution, and the dtype it therefore starts as
    _CHAINS = {'direct':  ((), TensorProto.FLOAT),                              # a plain initializer
               'dq':      (('DequantizeLinear',), TensorProto.INT8),            # what the pt2e flow emits
               'qdq':     (('QuantizeLinear', 'DequantizeLinear'), TensorProto.FLOAT),  # a QAT-style graph
               'dq_cast': (('DequantizeLinear', 'Cast'), TensorProto.INT8)}     # de-quantize plus a cast

    def _as_model(graph):
        "The graph wrapped in a model, which is what `onnx.checker` and `onnx.save` take"
        return helper.make_model(graph, opset_imports=[helper.make_opsetid('', 18)])

    def _conv_graph(kernel=(3, 5), op_type='Conv', chain='dq', kernel_shape=None, weight='initializer'):
        "Single-convolution graph whose weight reaches the node through `chain`, as `weight`"
        ops, dtype = _CHAINS[chain]
        dims = [f'd{i}' for i in range(len(kernel) + 2)]  # rank-correct; the values are irrelevant here
        array = np.zeros((8, 4) + tuple(kernel), np.int8 if dtype == TensorProto.INT8 else np.float32)
        inputs = [helper.make_tensor_value_info('x', TensorProto.FLOAT, dims)]
        initializers, nodes = [numpy_helper.from_array(np.array(0.05, np.float32), 'wscale')], []
        if weight == 'input':       # only known at run time: no shape to read anywhere
            inputs.append(helper.make_tensor_value_info('w', dtype, dims))
        elif weight == 'constant':  # emitted as a Constant node rather than as an initializer
            nodes.append(helper.make_node('Constant', [], ['w'],
                                          value=numpy_helper.from_array(array, 'wvalue')))
        else:
            initializers.append(numpy_helper.from_array(array, 'w'))
        source = 'w'
        for i, op in enumerate(ops):
            args = {'to': TensorProto.FLOAT} if op == 'Cast' else {}
            nodes.append(helper.make_node(op, [source] if op == 'Cast' else [source, 'wscale'],
                                          [f'w{i}'], **args))
            source = f'w{i}'
        attrs = {'kernel_shape': list(kernel_shape)} if kernel_shape else {}
        nodes.append(helper.make_node(op_type, ['x', source], ['y'], **attrs))
        graph = helper.make_graph(nodes, 'conv', inputs,
                                  [helper.make_tensor_value_info('y', TensorProto.FLOAT, dims)],
                                  initializer=initializers)
        onnx.checker.check_model(_as_model(graph))
        return graph

    def _kernel_shapes(graph):
        "The `kernel_shape` attribute of every convolution node, `None` where there is none"
        return [next((list(a.ints) for a in n.attribute if a.name == 'kernel_shape'), None)
                for n in graph.node if n.op_type in ('Conv', 'ConvTranspose')]

    # the weight is behind a DequantizeLinear: the pass follows it and writes the SPATIAL dims, in
    # order (an asymmetric kernel is what catches a transposed or mis-sliced shape)
    _g = _conv_graph()
    test_eq(_kernel_shapes(_g), [None])
    test_eq(_set_conv_kernel_shape(_g), 1)
    test_eq(_kernel_shapes(_g), [[3, 5]])
    onnx.checker.check_model(_as_model(_g))   # `kernel_shape` is optional, so writing it stays spec-legal
    test_eq(_set_conv_kernel_shape(_g), 0)    # idempotent

    # ...through a multi-node chain too — every operator in `_WEIGHT_SOURCE_OPS` is on one of these
    # paths, so dropping one from the tuple turns a case below red
    for _chain, _kernel in (('qdq', (3, 5)), ('dq_cast', (3, 5)), ('direct', (3, 5)), ('dq', (2, 3, 4))):
        _c = _conv_graph(kernel=_kernel, chain=_chain)
        test_eq(_set_conv_kernel_shape(_c), 1)
        test_eq(_kernel_shapes(_c), [list(_kernel)])

    # a weight emitted as a Constant node rather than as an initializer resolves the same way
    _gc = _conv_graph(weight='constant')
    test_eq(_set_conv_kernel_shape(_gc), 1)
    test_eq(_kernel_shapes(_gc), [[3, 5]])

    # a node that already carries the attribute is never touched, right or wrong
    _gk = _conv_graph(kernel_shape=[9, 9])
    test_eq(_set_conv_kernel_shape(_gk), 0)
    test_eq(_kernel_shapes(_gk), [[9, 9]])

    # only `Conv` is patched, an unreadable weight leaves its node alone, and an empty graph is a no-op
    test_eq(_set_conv_kernel_shape(_conv_graph(op_type='ConvTranspose')), 0)
    test_eq(_set_conv_kernel_shape(_conv_graph(weight='input')), 0)
    test_eq(_set_conv_kernel_shape(helper.make_graph([], 'empty', [], [])), 0)

    # --- a model whose weights live in an external data file survives the write path ---
    # shapes are read off the TensorProto, so the pass never materializes a weight, and the sidecar
    # is still the one the reloaded model reads from
    with tempfile.TemporaryDirectory() as _ext:
        _ext_path, _sidecar = Path(_ext)/'ext.onnx', Path(_ext)/'weights.bin'
        _ext_model = _as_model(_conv_graph())
        convert_model_to_external_data(_ext_model, all_tensors_to_one_file=True,
                                       location='weights.bin', size_threshold=0)
        onnx.save(_ext_model, str(_ext_path))
        assert _sidecar.exists(), "the fixture must actually put the weights in an external file"
        _ext_proto = onnx.load(str(_ext_path), load_external_data=False)  # what export_qdq loads
        test_eq(_set_conv_kernel_shape(_ext_proto.graph), 1)
        onnx.save(_ext_proto, str(_ext_path))
        _reloaded = onnx.load(str(_ext_path))                            # external data resolved again
        test_eq(_kernel_shapes(_reloaded.graph), [[3, 5]])
        test_eq(numpy_helper.to_array(
            next(_i for _i in _reloaded.graph.initializer if _i.name == 'w')).shape, (8, 4, 3, 5))

    # --- a refused export keeps nothing: not the graph, not its external-data file ---
    with tempfile.TemporaryDirectory() as _ext:
        _ext_path, _sidecar = Path(_ext)/'ext.onnx', Path(_ext)/'weights.bin'
        _ext_model = _as_model(_conv_graph())
        convert_model_to_external_data(_ext_model, all_tensors_to_one_file=True,
                                       location='weights.bin', size_threshold=0)
        onnx.save(_ext_model, str(_ext_path))
        with ExceptionExpected(ValueError, regex="opset 17"):
            _check_produced_opset(onnx.load(str(_ext_path), load_external_data=False), 17, _ext_path)
        assert not _ext_path.exists() and not _sidecar.exists(), \
            "'No file was kept' has to include the external-data file"

    # --- a graph that declares no default-domain opset gets its own message, not the converter story
    #     (which would be a false mechanism, and would advise `opset_version=None`) ---
    with tempfile.TemporaryDirectory() as _no:
        _no_path = Path(_no)/'no_opset.onnx'
        onnx.save(_as_model(_conv_graph()), str(_no_path))
        _no_proto = onnx.load(str(_no_path))
        del _no_proto.opset_import[:]
        with ExceptionExpected(ValueError, regex="declares no opset for the default ONNX domain"):
            _check_produced_opset(_no_proto, 18, _no_path)
        assert not _no_path.exists(), "a refused export must not leave a file behind"

if _HAS_PT2E and all(_has_package(p) for p in ('onnx', 'onnxscript', 'onnxruntime')):
    with tempfile.TemporaryDirectory() as _tmp:
        _cp = export_qdq(_qmodel, _sample, Path(_tmp)/'compat.onnx')
        _cm = onnx.load(str(_cp))

        # every Conv carries `kernel_shape`, each value read off that node's OWN weight — reached the
        # way a consumer would have to reach it, through the DequantizeLinear that hides it
        _convs = [_n for _n in _cm.graph.node if _n.op_type == 'Conv']
        assert _convs, "this post-condition is only meaningful on a graph that has Conv nodes"
        _dims = {_i.name: tuple(_i.dims) for _i in _cm.graph.initializer}
        _producers = {_o: _n for _n in _cm.graph.node for _o in _n.output}
        for _conv in _convs:
            _w = _conv.input[1]
            while _w not in _dims: _w = _producers[_w].input[0]
            test_eq(next((list(_a.ints) for _a in _conv.attribute if _a.name == 'kernel_shape'), None),
                    list(_dims[_w][2:]))
        onnx.checker.check_model(_cm)

        # and it is the ONLY thing that changed: strip the attributes back off and ONNX Runtime
        # returns byte-identical logits, over the same Q/DQ inventory
        _bare_path = Path(_tmp)/'no_kernel_shape.onnx'
        _bare = onnx.load(str(_cp))
        for _n in _bare.graph.node:
            if _n.op_type != 'Conv': continue
            for _i, _a in enumerate(_n.attribute):
                if _a.name == 'kernel_shape': del _n.attribute[_i]; break
        onnx.save(_bare, str(_bare_path))
        test_eq(sum(1 for _n in _bare.graph.node for _a in _n.attribute if _a.name == 'kernel_shape'), 0)
        test_eq(qdq_stats(_bare_path).as_dict(), qdq_stats(_cp).as_dict())
        _pre = ONNXModel(_bare_path)(_sample).numpy()
        _post = ONNXModel(_cp)(_sample).numpy()
        assert _pre.tobytes() == _post.tobytes(), \
            f"kernel_shape changed the arithmetic: max|Δlogit| = {np.abs(_pre - _post).max():.3e}"

        # the opset the file declares is the opset that was asked for — here, the one the exporter
        # emits; asking for an older one is the subject of the next cell
        test_eq([_o.version for _o in _cm.opset_import if _o.domain in ('', 'ai.onnx')], [18])

        # a model with no convolution is written exactly as the exporter emitted it
        _fc_only = Quantizer(backend='pt2e').quantize(
            nn.Sequential(nn.Flatten(), nn.Linear(3 * 16 * 16, 4)).eval(), _calib)
        _fc_graph = onnx.load(str(export_qdq(_fc_only, _sample, Path(_tmp)/'linear.onnx'))).graph
        test_eq([_n.op_type for _n in _fc_graph.node if _n.op_type == 'Conv'], [])
        test_eq(_set_conv_kernel_shape(_fc_graph), 0)

In [ ]:
# --- opset 17: the produced graph is REWRITTEN to it, and the file is checked and run before it counts ---
# (these run here, not in the exporter notebook, which the test suite does not execute; why the rewrite
#  has to happen at all is documented on `_lower_reducemean_axes` itself)
import fasterai.export.onnx_exporter as _exporter
from fasterai.export.onnx_exporter import (_declared_opset, _drop_unused_constant, _lower_produced_opset,
                                           _lower_reducemean_axes, _output_difference, _write_lowered)

if _has_package('onnx'):
    import onnx
    from onnx import TensorProto, helper, numpy_helper
    from onnx.external_data_helper import convert_model_to_external_data

    def _reduce_graph(axes=(-1, -2), axes_from='initializer', noop=0, extra=None, n_reduce=1,
                      unrunnable=False):
        "`n_reduce` ReduceMean nodes in the opset-18 spelling (`axes` as an INPUT), plus an optional node"
        inputs = [helper.make_tensor_value_info('x', TensorProto.FLOAT, [1, 2, 4, 4])]
        nodes, initializers = [], []
        array = np.array(axes, np.int64)
        if axes_from == 'initializer':
            initializers.append(numpy_helper.from_array(array, 'axes'))
        elif axes_from == 'constant':  # a producer may emit a constant as a node rather than as a tensor
            nodes.append(helper.make_node('Constant', [], ['axes'], name='node_axes',
                                          value=numpy_helper.from_array(array, 'axes_value')))
        else:                          # computed while the graph runs: there is nothing to move
            inputs.append(helper.make_tensor_value_info('axes_in', TensorProto.INT64, [len(axes)]))
            nodes.append(helper.make_node('Identity', ['axes_in'], ['axes'], name='node_axes'))
        for i in range(n_reduce):
            nodes.append(helper.make_node('ReduceMean', ['x', 'axes'], [f'm{i}'], name=f'node_mean{i}',
                                          keepdims=1, noop_with_empty_axes=noop))
        nodes.append(helper.make_node(extra or 'Identity', [f'm{n_reduce - 1}'],
                                      ['z' if unrunnable else 'y'],
                                      name=f'node_{(extra or "out").lower()}'))
        shape = [1, 2, 1, 1] if axes else [1, 1, 1, 1]  # reducing NO axis reduces them all
        if unrunnable:  # shapes that cannot broadcast: a checker infers none and takes the graph anyway
            initializers.append(numpy_helper.from_array(np.zeros((3, 5), np.float32), 'mismatched'))
            nodes.append(helper.make_node('Add', ['x', 'mismatched'], ['s'], name='node_add'))
            nodes.append(helper.make_node('Mul', ['z', 's'], ['y'], name='node_broken'))
            shape = [1, 2, 4, 4]  # what it would be, if it could run at all
        graph = helper.make_graph(nodes, 'reduce', inputs,
                                  [helper.make_tensor_value_info('y', TensorProto.FLOAT, shape)],
                                  initializer=initializers)
        model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 18)])
        onnx.checker.check_model(model)  # the fixture is a graph opset 18 accepts, to start with
        return model

    def _reduce_nodes(graph):
        "Every ReduceMean, as (inputs, attributes) — the two places the rewrite moves values between"
        return [(list(n.input), {a.name: (list(a.ints) if a.name == 'axes' else a.i) for a in n.attribute})
                for n in graph.node if n.op_type == 'ReduceMean']

    # the axes leave the input list and arrive in the attribute, and the constant that fed them is gone
    _m = _reduce_graph()
    test_eq(_reduce_nodes(_m.graph), [(['x', 'axes'], {'keepdims': 1, 'noop_with_empty_axes': 0})])
    test_eq(_lower_reducemean_axes(_m.graph), 1)
    test_eq(_reduce_nodes(_m.graph), [(['x'], {'keepdims': 1, 'axes': [-1, -2]})])
    test_eq([_i.name for _i in _m.graph.initializer], [])
    _m.opset_import[0].version = 17
    onnx.checker.check_model(_m)                   # ...and what comes out is a graph opset 17 accepts
    _frozen = _m.SerializeToString()
    test_eq(_lower_reducemean_axes(_m.graph), 0)   # idempotent: nothing left to move...
    test_eq(_m.SerializeToString(), _frozen)       # ...and nothing touched on the way through

    # axes emitted as a Constant NODE rather than as an initializer: the node itself is what has to go
    _mc = _reduce_graph(axes_from='constant')
    test_eq(_lower_reducemean_axes(_mc.graph), 1)
    test_eq(_reduce_nodes(_mc.graph), [(['x'], {'keepdims': 1, 'axes': [-1, -2]})])
    test_eq([_n.op_type for _n in _mc.graph.node], ['ReduceMean', 'Identity'])

    # one constant feeding two nodes survives the first rewrite and leaves with the second
    _m2 = _reduce_graph(n_reduce=2)
    test_eq(_lower_reducemean_axes(_m2.graph), 2)
    test_eq(_reduce_nodes(_m2.graph), [(['x'], {'keepdims': 1, 'axes': [-1, -2]})] * 2)
    test_eq([_i.name for _i in _m2.graph.initializer], [])

    # an EMPTY axes list means 'reduce every axis', which opset 17 says by leaving the attribute out
    _me = _reduce_graph(axes=())
    test_eq(_lower_reducemean_axes(_me.graph), 1)
    test_eq(_reduce_nodes(_me.graph), [(['x'], {'keepdims': 1})])
    # ...unless the graph says the opposite, which opset 17 has no way to express
    with ExceptionExpected(ValueError, regex="node_mean0"):
        _lower_reducemean_axes(_reduce_graph(axes=(), noop=1).graph)
    # ...and axes only known while the graph runs cannot become an attribute either
    with ExceptionExpected(ValueError, regex="node_mean0"):
        _lower_reducemean_axes(_reduce_graph(axes_from='runtime').graph)

    # --- the cleanup that follows removes a constant nothing reads, and only that ---
    _mk = _reduce_graph(n_reduce=2)
    test_eq(_drop_unused_constant(_mk.graph, 'axes'), False)     # two nodes still read it
    test_eq(_drop_unused_constant(_mk.graph, 'nothing'), False)  # no such name
    test_eq(_lower_reducemean_axes(_mk.graph), 2)
    test_eq(_drop_unused_constant(_mk.graph, 'axes'), False)     # already gone with the rewrite
    _exposed = helper.make_model(
        helper.make_graph([helper.make_node('Constant', [], ['c'], name='node_c',
                                            value=numpy_helper.from_array(np.zeros(2, np.float32), 'cv'))],
                          'exposed', [], [helper.make_tensor_value_info('c', TensorProto.FLOAT, [2])]),
        opset_imports=[helper.make_opsetid('', 18)])
    test_eq(_drop_unused_constant(_exposed.graph, 'c'), False)   # no reader, but the graph exposes it
    test_eq(len(_exposed.graph.node), 1)

    # --- what `_output_difference` calls identical, and what it does not ---
    _a, _b = np.zeros((2, 3), np.float32), np.zeros((2, 3), np.float32)
    test_eq(_output_difference([_a], [_b]), None)
    _b[0, 0] = 1e-6
    assert 'max |Δ|' in _output_difference([_a], [_b])
    assert 'became' in _output_difference([_a], [np.zeros((3, 2), np.float32)])
    assert 'became' in _output_difference([_a], [_a.astype(np.float64)])  # a dtype change is a change
    assert 'became' in _output_difference([_a], [_a, _a])

    with tempfile.TemporaryDirectory() as _tmp:
        _tmp = Path(_tmp)

        # a request the rewrite has no part in leaves the graph exactly as it was, for the opset check
        for _requested in (18, 19):
            _same = _reduce_graph()
            _before = _same.SerializeToString()
            test_eq(_lower_produced_opset(_same, _requested, _tmp/'never_written.onnx'), None)
            test_eq(_same.SerializeToString(), _before)

        # a graph with nothing to rewrite is still relabelled — and then left for the checker to judge
        _plain = helper.make_model(
            helper.make_graph([helper.make_node('Identity', ['x'], ['y'], name='node_out')], 'plain',
                              [helper.make_tensor_value_info('x', TensorProto.FLOAT, [2])],
                              [helper.make_tensor_value_info('y', TensorProto.FLOAT, [2])]),
            opset_imports=[helper.make_opsetid('', 18)])
        test_eq(_lower_produced_opset(_plain, 17, _tmp/'never_written.onnx'), 0)
        test_eq(_declared_opset(_plain), 17)

        # --- a refused rewrite keeps nothing: not the graph, not its external-data file ---
        _ext_path, _sidecar = _tmp/'runtime.onnx', _tmp/'runtime.bin'
        _ext_model = _reduce_graph(axes_from='runtime')
        _ext_model.graph.initializer.append(numpy_helper.from_array(np.zeros((8, 4), np.float32), 'spare'))
        convert_model_to_external_data(_ext_model, all_tensors_to_one_file=True,
                                       location='runtime.bin', size_threshold=0)
        onnx.save(_ext_model, str(_ext_path))
        assert _sidecar.exists(), "the fixture must actually put a tensor in an external file"
        with ExceptionExpected(ValueError, regex="node_mean0"):
            _lower_produced_opset(onnx.load(str(_ext_path), load_external_data=False), 17, _ext_path)
        assert not _ext_path.exists() and not _sidecar.exists(), \
            "'No file was kept' has to include the external-data file"

        # --- checking a lowered graph means RUNNING it, so without onnxruntime it is refused, not skipped
        _no_ort_path = _tmp/'no_ort.onnx'
        onnx.save(_reduce_graph(), str(_no_ort_path))
        _real_has_package = _exporter._has_package
        _exporter._has_package = lambda name: name != 'onnxruntime' and _real_has_package(name)
        try:
            with ExceptionExpected(ImportError, regex="onnxruntime"):
                _write_lowered(_reduce_graph(), _no_ort_path, torch.randn(1, 2, 4, 4), 18)
        finally:
            _exporter._has_package = _real_has_package
        assert not _no_ort_path.exists(), "a refusal must not leave the produced file behind"

        # --- the two ways a rewrite can end in a refusal, and neither answers for the other ---
        # `Mish` arrived in opset 18: the rewrite has no opinion about it, and CHECKING the file catches
        # it. A graph ONNX Runtime cannot run in the FIRST place fails in BOTH arms — so only running the
        # graph as produced, on its own and first, can say which of the two is actually at fault.
        if _has_package('onnxruntime'):
            _refusals = {}
            # the opset each arm is TOLD was produced differs, so the advice at the end of each message
            # can only be right if it comes from that argument — the rewritten proto declares 17 by now
            for _arm, _arm_model, _produced in (('rewrite', _reduce_graph(extra='Mish'), 18),
                                            ('produced', _reduce_graph(unrunnable=True), 99)):
                _arm_path = _tmp/f'{_arm}.onnx'
                onnx.save(_arm_model, str(_arm_path))
                test_eq(_lower_produced_opset(_arm_model, 17, _arm_path), 1)  # the ReduceMean lowers fine
                try: _write_lowered(_arm_model, _arm_path, torch.randn(1, 2, 4, 4), _produced)
                except ValueError as _error: _refusals[_arm] = str(_error)
                assert not _arm_path.exists() and not (_tmp/f'{_arm}.onnx.lowered').exists(), \
                    f"{_arm}: a refused rewrite keeps nothing"
            test_eq(sorted(_refusals), ['produced', 'rewrite'])  # both arms did refuse
            assert 'Mish' in _refusals['rewrite'], _refusals
            assert 'is not a graph ONNX accepts' in _refusals['rewrite'], _refusals
            assert 'could not run the graph it PRODUCED' in _refusals['produced'], _refusals
            # ...and neither message is the other one: a graph that was broken before the rewrite must
            # never be reported as a rewrite that broke it
            assert 'PRODUCED' not in _refusals['rewrite'], _refusals
            assert 'is not a graph ONNX accepts' not in _refusals['produced'], _refusals
            # both say how to get the graph as it was produced, like their siblings above
            assert 'opset_version=18' in _refusals['rewrite'], _refusals
            assert 'opset_version=99' in _refusals['produced'], _refusals
        test_eq(sorted(_p.name for _p in _tmp.iterdir()), [])  # every refusal above kept nothing at all

if _HAS_PT2E and all(_has_package(p) for p in ('onnx', 'onnxscript', 'onnxruntime')):
    with tempfile.TemporaryDirectory() as _tmp:
        _tmp = Path(_tmp)
        _p18 = export_qdq(_qmodel, _sample, _tmp/'opset18.onnx')
        _p17 = export_qdq(_qmodel, _sample, _tmp/'opset17.onnx', opset_version=17)
        _m18, _m17 = (onnx.load(str(_p)) for _p in (_p18, _p17))

        # the file declares the opset that was asked for...
        test_eq([_o.version for _o in _m18.opset_import if _o.domain in ('', 'ai.onnx')], [18])
        test_eq([_o.version for _o in _m17.opset_import if _o.domain in ('', 'ai.onnx')], [17])
        # ...because the operator that stood in the way was rewritten, not because it was relabelled
        _r18, _r17 = (_reduce_nodes(_m.graph) for _m in (_m18, _m17))
        assert _r18 and all(len(_inputs) == 2 for _inputs, _attrs in _r18), _r18
        assert _r17 and all(len(_inputs) == 1 for _inputs, _attrs in _r17), _r17
        # ...with the axes the constant held, in order
        _axes_of = {_i.name: [int(_v) for _v in numpy_helper.to_array(_i).reshape(-1)]
                    for _i in _m18.graph.initializer}
        test_eq([_attrs['axes'] for _inputs, _attrs in _r17],
                [_axes_of[_n.input[1]] for _n in _m18.graph.node if _n.op_type == 'ReduceMean'])
        # ...and the constant that fed them went with it, without anything taking its place
        test_eq({_i.name for _i in _m18.graph.initializer} - {_i.name for _i in _m17.graph.initializer},
                {_n.input[1] for _n in _m18.graph.node if _n.op_type == 'ReduceMean'})
        test_eq({_i.name for _i in _m17.graph.initializer} - {_i.name for _i in _m18.graph.initializer},
                set())

        # ONNX reads the file at the opset it now declares, over the same Q/DQ inventory...
        onnx.checker.check_model(str(_p17))
        test_eq(qdq_stats(_p17).as_dict(), qdq_stats(_p18).as_dict())
        # ...the `kernel_shape` pass ran on it too (both post-conditions hold, on the same file)...
        def _kernel_shapes(_m):
            return [next((list(_a.ints) for _a in _n.attribute if _a.name == 'kernel_shape'), None)
                    for _n in _m.graph.node if _n.op_type == 'Conv']

        assert all(_kernel_shapes(_m17)), _kernel_shapes(_m17)
        test_eq(_kernel_shapes(_m17), _kernel_shapes(_m18))
        # ...and ONNX Runtime returns BYTE-IDENTICAL logits: the rewrite changed a spelling, not a graph
        _pre, _post = (ONNXModel(_p)(_sample).numpy() for _p in (_p18, _p17))
        assert _pre.tobytes() == _post.tobytes(), \
            f"the lowering changed the arithmetic: max|Δlogit| = {np.abs(_pre - _post).max():.3e}"
        # and the export leaves no scratch file behind
        test_eq(sorted(_p.name for _p in _tmp.iterdir()), ['opset17.onnx', 'opset18.onnx'])

In [ ]:
# --- qdq_placement: the grammar, and the quantizer it builds ---
# (every test of this axis lives here rather than in the exporter notebook, which the suite does not
#  execute: `skip_exec: true`)
from fasterai.core.precision import QDQ_PLACEMENTS

test_eq(QDQ_PLACEMENTS, ('per_op', 'skip_conv_add'))
# it joins the precision arguments, so it cannot be layered on a hand-written qconfig mapping either
assert 'qdq_placement' in _GRAMMAR_ARGS, _GRAMMAR_ARGS

# The default is the placement pt2e has always produced, and asking for it explicitly is the same ask.
# The grammar is pure python, so these resolve the same way on a runner whose torch build has no pt2e.
test_eq(_resolve_spec('pt2e').qdq_placement, 'per_op')
test_eq(_resolve_spec('pt2e', qdq_placement='per_op'), _resolve_spec('pt2e'))
test_eq(_resolve_spec('pt2e', qdq_placement='skip_conv_add').qdq_placement, 'skip_conv_add')
assert _resolve_spec('pt2e', qdq_placement='skip_conv_add') != _resolve_spec('pt2e')
test_eq(Quantizer().spec.qdq_placement, None)   # the FX flow places nothing, so it records nothing

# --- refusals, from the user's side (the grammar resolves before the backend is looked for, so these
#     raise the same way on a runner whose torch build ships no pt2e) ---
_refused("names no placement at all", backend='x86', qdq_placement='skip_conv_add')
_refused("names no placement at all", backend='x86', qdq_placement='per_op')  # even the default one
_refused("The backend\\(s\\) that can: \\['pt2e'\\]", backend='qnnpack', qdq_placement='skip_conv_add')
_refused("Unknown qdq_placement 'fuse_residuals'", backend='pt2e', qdq_placement='fuse_residuals')
_refused("`qdq_placement` must be one of ", exc=TypeError, backend='pt2e', qdq_placement=True)
_refused("Keep one of the two", qconfig_mapping=get_default_qconfig_mapping('x86'),
         qdq_placement='skip_conv_add')

# the option says what it costs where a caller reads it: opt-in, and unmeasured on a trained model
_placement_doc = Quantizer.__init__.__doc__
assert 'opt-in' in _placement_doc and 'UNMEASURED' in _placement_doc, _placement_doc

if _HAS_PT2E:
    # ...and the `Quantizer` that will apply it carries the same resolution
    test_eq(Quantizer(backend='pt2e').spec.qdq_placement, 'per_op')
    test_eq(Quantizer(backend='pt2e', qdq_placement='per_op').spec, Quantizer(backend='pt2e').spec)
    test_eq(Quantizer(backend='pt2e', qdq_placement='skip_conv_add').spec.qdq_placement, 'skip_conv_add')

    # 'per_op' builds the very quantizer this backend has always built — no wrapper at all
    test_eq(type(_pt2e_quantizer()), XNNPACKQuantizer)
    test_eq(type(_pt2e_quantizer(Quantizer(backend='pt2e').spec)), XNNPACKQuantizer)
    _wrapped = _pt2e_quantizer(Quantizer(backend='pt2e', qdq_placement='skip_conv_add').spec)
    test_eq(type(_wrapped), _PlacementQuantizer)
    test_eq(_wrapped.placement, 'skip_conv_add')
    assert isinstance(_wrapped, _TorchQuantizer), "the wrapper has to BE a torch quantizer"

    # `prepare_pt2e` reads exactly four things off a quantizer, and all four are delegated — the
    # callback as an ATTRIBUTE, because that is how it is read (bound now, called later)
    test_eq(_wrapped.prepare_obs_or_fq_callback, _wrapped.inner.prepare_obs_or_fq_callback)
    for _name in ('transform_for_annotation', 'annotate', 'validate'):
        assert callable(getattr(_wrapped, _name)), _name
    # ...and the precision underneath is untouched by the placement: it is a different axis
    test_eq(_wrapped.inner.global_config.weight.qscheme, torch.per_channel_symmetric)
    test_eq(_pt2e_quantizer(Quantizer(backend='pt2e', qscheme='per_tensor',
                                      qdq_placement='skip_conv_add').spec
                            ).inner.global_config.weight.qscheme, torch.per_tensor_symmetric)
    with warnings.catch_warnings():   # `symmetric=False` warns about the zero-points it keeps
        warnings.simplefilter('ignore')
        _affine_spec = Quantizer(backend='pt2e', symmetric=False,
                                 qdq_placement='skip_conv_add').spec
    test_eq(_pt2e_quantizer(_affine_spec).inner.global_config.input_activation.qscheme,
            torch.per_tensor_affine)

    # a spec pickled before this field existed unpickles with the slot UNSET, not with the default:
    # the builder reads it the way the export post-condition does, and gets the ordinary quantizer
    _stale = QuantSpec.__new__(QuantSpec)
    for _field, _value in Quantizer(backend='pt2e').spec.as_dict().items():
        if _field != 'qdq_placement': object.__setattr__(_stale, _field, _value)
    with ExceptionExpected(AttributeError): _stale.qdq_placement
    test_eq(type(_pt2e_quantizer(_stale)), XNNPACKQuantizer)
else:
    # ...and on a torch build with no pt2e flow, asking for the backend that has this axis fails
    # loudly, naming what is missing — the placement is never quietly dropped
    with ExceptionExpected(ImportError, regex="quantize_pt2e"):
        Quantizer(backend='pt2e', qdq_placement='skip_conv_add')

In [ ]:
# --- qdq_placement='skip_conv_add': what it does to the graph, on models built to probe it ---
class _TinyResidual(nn.Module):
    "One residual block, `x + branch(x)` — the shape this placement is named for"
    # The head is deliberately un-pooled: an average over 256 positions hides a sub-step difference,
    # and the point of the arms below is to be able to SEE the arithmetic change.
    def __init__(self, n_classes=4):
        super().__init__()
        self.stem = nn.Conv2d(3, 8, 3, padding=1)
        self.branch = nn.Sequential(nn.Conv2d(8, 8, 3, padding=1), nn.BatchNorm2d(8))
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(8 * 16 * 16, n_classes))
    def forward(self, x):
        x = torch.relu(self.stem(x))
        return self.head(torch.relu(x + self.branch(x)))


class _TwoBranches(nn.Module):
    "Both addends are convolutions, as in a downsample block: only ONE of the two edges may be cleared"
    def __init__(self):
        super().__init__()
        self.left, self.right = nn.Conv2d(3, 8, 3, padding=1), nn.Conv2d(3, 8, 1)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(8 * 16 * 16, 4))
    def forward(self, x): return self.head(torch.relu(self.left(x) + self.right(x)))


class _SharedConv(nn.Module):
    "The convolution reaching the addition feeds a second consumer, so its pair is not this edge's"
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 8, 3, padding=1)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(8 * 16 * 16, 4))
    def forward(self, x):
        c = self.conv(x)
        return self.head(torch.relu(c + c.mean(dim=1, keepdim=True)))


class _NoConvAdd(nn.Module):
    "Its addition adds two things no convolution produced"
    def __init__(self):
        super().__init__(); self.left, self.right = nn.Linear(8, 8), nn.Linear(8, 8)
    def forward(self, x): return self.left(x) + self.right(x)


def _module_path(node):
    "Where in the SOURCE model a captured node came from — stable across torch's node renamings"
    stack = node.meta.get('nn_module_stack') or {}
    return list(stack.values())[-1][0] if stack else None


def _add_sources(graph_module):
    "For every addition, what each addend IS: a dequantize node, or the operator itself"
    return [tuple('dequantize' if 'dequantize_per' in str(getattr(a, 'target', '')) else
                  str(getattr(a, 'target', a)) for a in n.args[:2])
            for n in graph_module.graph.nodes
            if n.op == 'call_function' and n.target in _ADD_TARGETS]


def _n_qdq(graph_module, kind):
    "How many quantize/dequantize calls of one kind a converted graph holds"
    return sum(1 for n in graph_module.graph.nodes
               if n.op == 'call_function' and kind in str(n.target))


def _qdq_params(graph_module):
    "Every q/dq node's (scale, zero-point), keyed by the ORIGINAL tensor it quantizes"
    # Keyed by the first node upstream that is NOT a q/dq node, because that name is the one the two
    # arms share: the `quantize_per_tensor_default_N` counter shifts as soon as one pair is removed,
    # and keying on it would compare two DIFFERENT tensors and call them equal.
    def _source(node):
        current = node.args[0]
        while isinstance(current, torch.fx.Node) and 'quantize_per' in str(current.target):
            current = current.args[0]
        return getattr(current, 'name', str(current))

    def _value(arg):
        if not isinstance(arg, torch.fx.Node): return arg
        return tuple(getattr(graph_module, arg.target).detach().reshape(-1).tolist())

    params = {}
    for node in graph_module.graph.nodes:
        if node.op != 'call_function' or 'quantize_per' not in str(node.target): continue
        params.setdefault((_source(node), str(node.target)), set()).add(
            tuple(_value(a) for a in node.args[1:3]))
    return params


def _graph_digest(graph_module):
    "Node targets, node arguments and every buffer, hashed — what 'the same artifact' means here"
    digest = hashlib.sha256()
    for node in graph_module.graph.nodes:  # the arguments carry the per-tensor scales, as literals
        digest.update(f"{node.op}|{node.target}|{[str(a) for a in node.args]}\n".encode())
    for name, buffer in sorted(graph_module.named_buffers()):
        digest.update(name.encode()); digest.update(buffer.detach().cpu().numpy().tobytes())
    return digest.hexdigest()


if _HAS_PT2E:
    torch.manual_seed(0)
    _res_calib = [torch.randn(4, 3, 16, 16) for _ in range(3)]
    _res_model = _TinyResidual().eval()
    _per_op = Quantizer(backend='pt2e').quantize(_res_model, _res_calib)
    _skip = Quantizer(backend='pt2e', qdq_placement='skip_conv_add').quantize(_res_model, _res_calib)

    # RED -> GREEN, read off the graph: per_op puts a dequantize node on BOTH addends...
    test_eq(_add_sources(_per_op), [('dequantize', 'dequantize')])
    # ...skip_conv_add leaves the residual branch's convolution reaching the addition itself
    test_eq(_add_sources(_skip), [('dequantize', 'aten.conv2d.default')])
    # ...which is exactly one pair fewer, and no other kind of pair touched
    for _kind, _delta in (('quantized_decomposed.quantize_per_tensor', 1),
                          ('quantized_decomposed.dequantize_per_tensor', 1),
                          ('quantize_per_channel', 0), ('dequantize_per_channel', 0)):
        test_eq((_kind, _n_qdq(_per_op, _kind) - _n_qdq(_skip, _kind)), (_kind, _delta))

    # NOT A NO-OP: an implementation that quietly did nothing would answer identically here
    _res_probes = [torch.randn(4, 3, 16, 16) for _ in range(4)]
    with torch.no_grad():
        _arm_delta = max(float((_per_op(p) - _skip(p)).abs().max()) for p in _res_probes)
    assert _arm_delta > 0, "the two placements compute the same thing: the pass did nothing at all"

    # ONE VARIABLE: every pair the skip arm KEEPS carries the scale the per_op arm gave it, to the
    # last bit. That is the PTQ claim — an observer records a range, it does not round — measured on
    # the artifact rather than argued from the observer classes. (PTQ only: under QAT the
    # fake-quantize modules alter the tensors, so removing one moves the statistics downstream.)
    _params_per_op, _params_skip = _qdq_params(_per_op), _qdq_params(_skip)
    test_eq([k for k in set(_params_per_op) & set(_params_skip)
             if _params_per_op[k] != _params_skip[k]], [])
    test_eq(set(_params_skip) - set(_params_per_op), set())   # the skip arm invents no new scale
    assert set(_params_per_op) - set(_params_skip), "no pair was removed: this check proves nothing"

    # GOLDEN: the default artifact, byte for byte, whether or not the default is spelled out
    _default_digest = _graph_digest(Quantizer(backend='pt2e').quantize(_res_model, _res_calib))
    test_eq(_graph_digest(Quantizer(backend='pt2e', qdq_placement='per_op').quantize(
        _res_model, _res_calib)), _default_digest)
    test_eq(_graph_digest(_per_op), _default_digest)
    assert _graph_digest(_skip) != _default_digest, "the skip arm produced the default artifact"

    # --- the guards ---
    # a downsample-shaped block adds two convolutions: exactly ONE edge is cleared, the FIRST addend,
    # and the second keeps its pair — an addition with no quantized input at all is not a placement
    _two = Quantizer(backend='pt2e', qdq_placement='skip_conv_add').quantize(
        _TwoBranches().eval(), _res_calib)
    test_eq(_add_sources(_two), [('aten.conv2d.default', 'dequantize')])
    _two_add = next(n for n in _two.graph.nodes
                    if n.op == 'call_function' and n.target in _ADD_TARGETS)
    test_eq(_module_path(_two_add.args[0]), 'left')   # the first addend, not the other convolution

    # the convolution reaching the addition feeds something else too: its pair is shared, so nothing
    # matches — and a placement that matched nothing refuses rather than being recorded as applied
    with ExceptionExpected(ValueError, regex="found no conv"):
        Quantizer(backend='pt2e', qdq_placement='skip_conv_add').quantize(
            _SharedConv().eval(), _res_calib)
    # an addition fed by two Linear layers: same refusal, and it counts the additions it looked at
    with ExceptionExpected(ValueError, regex="1 add node"):
        Quantizer(backend='pt2e', qdq_placement='skip_conv_add').quantize(
            _NoConvAdd().eval(), [torch.randn(4, 8)])
    # a model with no addition at all
    with ExceptionExpected(ValueError, regex="0 add node"):
        Quantizer(backend='pt2e', qdq_placement='skip_conv_add').quantize(_TinyConvNet().eval(), _calib)
    # ...and the refusal happens BEFORE anything is recorded: no model comes back
    test_eq(quant_spec(_TinyConvNet()), None)

    # a tensor another spec is defined against keeps its pair: removing the qspec a
    # SharedQuantizationSpec points at would leave that reference dangling, which is a broken graph
    # rather than a different placement
    class _SharedSpecProbe(_PlacementQuantizer):
        "Runs the real annotation, then asks the matcher the same question once more with a shared spec"
        def annotate(self, model):
            model = self.inner.annotate(model)
            self.matched = [(add.name, addend.name) for add, addend in _conv_add_edges(model)]
            _add_node, _addend = _conv_add_edges(model)[0]
            _annotation = _node_annotation(_add_node)
            _kept, _annotation.output_qspec = (_annotation.output_qspec,
                                               SharedQuantizationSpec(_addend))
            self.guarded = [(add.name, addend.name) for add, addend in _conv_add_edges(model)]
            _annotation.output_qspec = _kept   # put the annotation back before preparing on it
            return model

    _probe = _SharedSpecProbe(_pt2e_quantizer(_resolve_spec('pt2e')), 'skip_conv_add')
    prepare_pt2e(torch.export.export(_TinyResidual().eval(), (_res_calib[0],)).module(), _probe)
    test_eq(len(_probe.matched), 1)   # the edge is there, and the ordinary annotation finds it...
    test_eq(_probe.guarded, [])       # ...and it is left alone once something shares its scale

    # provenance round-trip: the placement travels with the model that has it
    test_eq(quant_spec(_skip).qdq_placement, 'skip_conv_add')
    test_eq(quant_spec(_per_op).qdq_placement, 'per_op')
    with ExceptionExpected(AttributeError): quant_spec(_skip).qdq_placement = 'per_op'

In [ ]:
# --- the three graph shapes the matcher has to get right, on real models ---
# A rule reading `args[0]` and expecting a bare convolution there matches 8/8 on ResNet-18 and NOTHING
# on the other two: MobileNetV2's residual branch is the SECOND addend (its block is `x + conv(x)`,
# and `x` has two users), and a QAT graph is annotated BEFORE conv+bn fusion, so the qspec sits on the
# batch-norm. Both would leave the option silently inapplicable on those models.
if _HAS_PT2E:
    from torchvision.models import mobilenet_v2, resnet18

    def _unobserved_add_inputs(prepared):
        "Addition inputs a PREPARED graph reads straight from an operator, with no observer between"
        # `call_function` only: a batch-norm's `running_var + eps` reads a get_attr parameter, and
        # that addition is not one this grammar is about.
        return [a for n in prepared.graph.nodes
                if n.op == 'call_function' and n.target in _ADD_TARGETS
                for a in n.args[:2] if isinstance(a, torch.fx.Node) and a.op == 'call_function']

    _blocks = [f'layer{stage}.{block}' for stage in (1, 2, 3, 4) for block in (0, 1)]
    _mobile = [f'features.{i}' for i in (3, 5, 6, 8, 9, 10, 12, 13, 15, 16)]
    for _label, _factory, _method, _expected in (
            ('resnet18 PTQ', resnet18, 'static', [f'{b}.conv2' for b in _blocks]),
            ('resnet18 QAT', resnet18, 'qat', [f'{b}.bn2' for b in _blocks]),
            ('mobilenet_v2 PTQ', mobilenet_v2, 'static', [f'{b}.conv.2' for b in _mobile]),
            ('mobilenet_v2 QAT', mobilenet_v2, 'qat', [f'{b}.conv.3' for b in _mobile])):
        _shape_sample = torch.randn(2, 3, 64, 64)
        _base = _prepare_pt2e(_factory(weights=None), _shape_sample, _resolve_spec('pt2e', _method))
        _moved = _prepare_pt2e(_factory(weights=None), _shape_sample,
                               _resolve_spec('pt2e', _method, qdq_placement='skip_conv_add'))
        # the ordinary placement observes every addend: there is no edge to find in that arm
        test_eq((_label, _unobserved_add_inputs(_base)), (_label, []))
        # ...and the skip arm leaves exactly the residual branches unquantized, named by the SOURCE
        # module they came from: on ResNet-18 every block's second convolution and never a downsample
        # (the downsample convolution is the second addend, and it keeps its pair)
        test_eq((_label, [_module_path(a) for a in _unobserved_add_inputs(_moved)]),
                (_label, _expected))

In [ ]:
# --- the file has to carry the placement the model records, and is checked for it ---
# (here rather than in the exporter notebook, which the suite does not execute: `skip_exec: true`)
from fasterai.export.onnx_exporter import (_check_produced_placement, _direct_conv_add_edges,
                                           _producers)

if _has_package('onnx'):
    import onnx
    from onnx import TensorProto, helper, numpy_helper

    def _conv_add_graph(through_qdq: bool, weight='initializer', through_relu=False,
                        relu_shared=False):
        "A Conv feeding an Add — directly, through its own activation, or through a Q/DQ pair"
        dims = ['n', 'c', 'h', 'w']
        inputs = [helper.make_tensor_value_info(name, TensorProto.FLOAT, dims) for name in ('x', 'r')]
        initializers = [numpy_helper.from_array(np.array(0.05, np.float32), 'scale'),
                        numpy_helper.from_array(np.array(0, np.int8), 'zp')]
        array, nodes, outputs = np.zeros((4, 4, 3, 3), np.float32), [], ['y']
        if weight == 'constant':  # a producer may emit a weight as a Constant node, not a tensor
            nodes.append(helper.make_node('Constant', [], ['w'],
                                          value=numpy_helper.from_array(array, 'wvalue')))
        else:
            initializers.append(numpy_helper.from_array(array, 'w'))
        nodes.append(helper.make_node('Conv', ['x', 'w'], ['c'], kernel_shape=[3, 3]))
        source = 'c'
        if through_relu:
            nodes.append(helper.make_node('Relu', ['c'], ['a']))
            source = 'a'
            if relu_shared:  # the activation is read somewhere else too, so it is not the conv's own
                nodes.append(helper.make_node('Identity', ['a'], ['spare']))
                outputs.append('spare')
        if through_qdq:
            nodes += [helper.make_node('QuantizeLinear', [source, 'scale', 'zp'], ['q']),
                      helper.make_node('DequantizeLinear', ['q', 'scale', 'zp'], ['d'])]
            source = 'd'
        nodes.append(helper.make_node('Add', [source, 'r'], ['y']))
        graph = helper.make_graph(nodes, 'convadd', inputs,
                                  [helper.make_tensor_value_info(n, TensorProto.FLOAT, dims)
                                   for n in outputs],
                                  initializer=initializers)
        model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 18)])
        onnx.checker.check_model(model)
        return model

    # the edge is counted when the addition reads the convolution, and not when a pair sits between
    test_eq(_direct_conv_add_edges(_conv_add_graph(through_qdq=False).graph), 1)
    test_eq(_direct_conv_add_edges(_conv_add_graph(through_qdq=True).graph), 0)
    # ...however the weight got into the graph: the count reads the NODES, not the constants
    test_eq(_direct_conv_add_edges(_conv_add_graph(False, weight='constant').graph), 1)
    test_eq(_direct_conv_add_edges(_conv_add_graph(True, weight='constant').graph), 0)
    test_eq(_direct_conv_add_edges(helper.make_graph([], 'empty', [], [])), 0)
    # the convolution's own activation is walked back through — it is part of the partition the
    # annotation-time matcher cleared, so a file carrying `Conv -> Relu -> Add` carries the edge
    test_eq(_direct_conv_add_edges(_conv_add_graph(False, through_relu=True).graph), 1)
    test_eq(_direct_conv_add_edges(_conv_add_graph(True, through_relu=True).graph), 0)
    # ...but only while the activation feeds that addition and nothing else, which is the same
    # single-user rule the matcher applies: shared, it is no longer one convolution's private result
    test_eq(_direct_conv_add_edges(
        _conv_add_graph(False, through_relu=True, relu_shared=True).graph), 0)

    with tempfile.TemporaryDirectory() as _tmp:
        _tmp = Path(_tmp)
        # `qdq_stats` reports it off the file, beside the Q/DQ counts
        for _through, _expected in ((False, 1), (True, 0)):
            _stats_path = _tmp/f'stats_{_through}.onnx'
            onnx.save(_conv_add_graph(through_qdq=_through), str(_stats_path))
            test_eq(qdq_stats(_stats_path).n_unquantized_conv_add, _expected)
            test_eq(qdq_stats(_stats_path).n_quantize, 0 if not _through else 1)

        # --- the post-condition: a produced file that CONTRADICTS the spec is refused ---
        _skip_model = nn.Linear(4, 4)
        setattr(_skip_model, SPEC_ATTR, _resolve_spec('pt2e', qdq_placement='skip_conv_add'))
        _bad_path, _bad = _tmp/'contradiction.onnx', _conv_add_graph(through_qdq=True)
        onnx.save(_bad, str(_bad_path))
        with ExceptionExpected(ValueError, regex="contradicts the model it exported"):
            _check_produced_placement(_bad, _skip_model, _bad_path)
        assert not _bad_path.exists(), "a refused export must not leave a file behind"

        # ...and a file that does carry the edge is kept
        _good_path, _good = _tmp/'kept.onnx', _conv_add_graph(through_qdq=False)
        onnx.save(_good, str(_good_path))
        _check_produced_placement(_good, _skip_model, _good_path)
        assert _good_path.exists()

        # the check reads the SPEC, and only the one value it is about. An untagged model, a per_op
        # model, and a spec pickled before this field existed — a frozen slots dataclass unpickles
        # with the SLOT UNSET, not with the default — all pass a file the skip spec would refuse.
        _stale_spec = QuantSpec.__new__(QuantSpec)
        with ExceptionExpected(AttributeError): _stale_spec.qdq_placement
        _per_op_model, _stale_model = nn.Linear(4, 4), nn.Linear(4, 4)
        setattr(_per_op_model, SPEC_ATTR, _resolve_spec('pt2e'))
        setattr(_stale_model, SPEC_ATTR, _stale_spec)
        for _label, _probe_model in (('untagged', nn.Linear(4, 4)), ('per_op', _per_op_model),
                                     ('pickled before the field', _stale_model)):
            _kept_path = _tmp/'untouched.onnx'
            onnx.save(_conv_add_graph(through_qdq=True), str(_kept_path))
            _check_produced_placement(onnx.load(str(_kept_path)), _probe_model, _kept_path)
            assert _kept_path.exists(), _label

# --- and end to end: the two arms, exported, and each file checked against ITS OWN model ---
if _HAS_PT2E and all(_has_package(p) for p in ('onnx', 'onnxscript', 'onnxruntime')):
    # Measured here (torch 2.9.1, onnxruntime CPU): the same batch through the exported file and
    # through the quantized PyTorch model agrees to 0.0e+00 on both arms, against 5.1e-01 when the
    # ONNX arm is fed a DIFFERENT batch. _RES_ATOL sits between the two, and the last assertion of
    # each arm proves it: a wrong-input harness turns this test red.
    _RES_ATOL = 5e-2
    with tempfile.TemporaryDirectory() as _tmp:
        _tmp = Path(_tmp)
        _res_sample, _res_other = torch.randn(4, 3, 16, 16), torch.randn(4, 3, 16, 16)
        _res_files = {_arm: export_qdq(_arm_model, _res_sample, _tmp/f'{_arm}.onnx')
                      for _arm, _arm_model in (('per_op', _per_op), ('skip', _skip))}
        _res_stats = {_arm: qdq_stats(_path) for _arm, _path in _res_files.items()}

        # the placement is visible in the FILE, which is where a consumer reads it
        test_eq(_res_stats['per_op'].n_unquantized_conv_add, 0)
        test_eq(_res_stats['skip'].n_unquantized_conv_add, 1)
        # exactly one Q/DQ pair fewer, over the same per-channel weights, both fully symmetric
        test_eq(_res_stats['per_op'].n_quantize - _res_stats['skip'].n_quantize, 1)
        test_eq(_res_stats['per_op'].n_dequantize - _res_stats['skip'].n_dequantize, 1)
        test_eq(_res_stats['per_op'].n_per_channel, _res_stats['skip'].n_per_channel)
        for _arm, _arm_stats in _res_stats.items():
            test_eq((_arm, _arm_stats.n_nonzero_zero_point), (_arm, 0))

        # parity is PER ARM: the arms compute different things on purpose, so comparing one arm's
        # file against the other arm's model would only measure the option
        for _arm, _arm_model in (('per_op', _per_op), ('skip', _skip)):
            _arm_session = ONNXModel(_res_files[_arm])
            with torch.no_grad(): _torch_logits = _arm_model(_res_sample).numpy()
            _onnx_logits = _arm_session(_res_sample).numpy()
            assert np.allclose(_torch_logits, _onnx_logits, atol=_RES_ATOL), \
                f"{_arm}: max|Δlogit| = {np.abs(_torch_logits - _onnx_logits).max():.3e}"
            assert not np.allclose(_torch_logits, _arm_session(_res_other).numpy(), atol=_RES_ATOL), \
                f"{_arm}: the parity check cannot tell one input batch from another"

        # --- a residual branch that ends in an ACTIVATION ---
        # The pass clears that edge at annotation time (the qspec sits on the ReLU, which is what
        # `_annotate_conv_relu` does), and the produced file then reads `Conv -> Relu -> Add`. Read
        # off the addition's DIRECT producer only, the post-condition found no edge and deleted a
        # perfectly good file, with a message that was factually wrong about it.
        class _ReluBranch(nn.Module):
            "Its residual branch ends in an activation: conv -> relu -> add"
            def __init__(self):
                super().__init__()
                self.stem = nn.Conv2d(3, 8, 3, padding=1)
                self.branch = nn.Conv2d(8, 8, 3, padding=1)
                self.head = nn.Sequential(nn.Flatten(), nn.Linear(8 * 16 * 16, 4))
            def forward(self, x):
                x = torch.relu(self.stem(x))
                return self.head(x + torch.relu(self.branch(x)))

        _relu_model = Quantizer(backend='pt2e', qdq_placement='skip_conv_add').quantize(
            _ReluBranch().eval(), _res_calib)
        _relu_path = export_qdq(_relu_model, _res_sample, _tmp/'relu_branch.onnx')
        assert _relu_path.exists(), "the export of a graph that DOES carry the edge was refused"
        test_eq(qdq_stats(_relu_path).n_unquantized_conv_add, 1)
        # ...and the edge really is behind the activation: the addition's own producer is the Relu
        _relu_graph = onnx.load(str(_relu_path)).graph
        test_eq(sorted(_producers(_relu_graph)[_i].op_type
                       for _n in _relu_graph.node if _n.op_type == 'Add' for _i in _n.input),
                ['DequantizeLinear', 'Relu'])

        # ...while a produced file that does NOT carry the edge is still refused, keeping nothing:
        # strip it out of that same graph post hoc by giving the activation a second consumer, which
        # is exactly what takes it out of one convolution's private partition
        _stripped = onnx.load(str(_relu_path))
        _stripped_add = next(_n for _n in _stripped.graph.node if _n.op_type == 'Add')
        _stripped_relu = next(_i for _i in _stripped_add.input
                              if _producers(_stripped.graph)[_i].op_type == 'Relu')
        _stripped.graph.node.append(helper.make_node('Identity', [_stripped_relu], ['spare']))
        _stripped.graph.output.append(
            helper.make_tensor_value_info('spare', TensorProto.FLOAT, None))
        test_eq(_direct_conv_add_edges(_stripped.graph), 0)
        _stripped_path = _tmp/'stripped.onnx'
        onnx.save(_stripped, str(_stripped_path))
        with ExceptionExpected(ValueError, regex="contradicts the model it exported"):
            _check_produced_placement(_stripped, _relu_model, _stripped_path)
        assert not _stripped_path.exists(), "a refused export must not leave a file behind"

In [ ]:
# --- verify_qdq says when its own answer is vacuous ---
# A perfect agreement on a model that answers one class to everything is not evidence: the two arms
# below pin that the warning fires exactly when the reference predictions are degenerate.
if all(_has_package(p) for p in ('onnx', 'onnxruntime')):
    from fasterai.export.onnx_exporter import export_onnx

    class _Constant(nn.Module):
        "Every input gets the same logits, so its argmax cannot distinguish anything"
        def __init__(self):
            super().__init__()
            self.fc = nn.Linear(4, 3)
            nn.init.zeros_(self.fc.weight); nn.init.zeros_(self.fc.bias)
        def forward(self, x): return self.fc(x)

    class _Passthrough(nn.Module):
        "Its prediction follows the input, so the argmax varies from one row to the next"
        def forward(self, x): return x * 1.0

    with tempfile.TemporaryDirectory() as _tmp:
        _probe = torch.randn(8, 4)

        _const_path = export_onnx(_Constant().eval(), _probe, Path(_tmp)/'constant.onnx',
                                  dynamic_batch=False, optimize=False)
        with warnings.catch_warnings(record=True) as _caught:
            warnings.simplefilter('always')
            test_eq(verify_qdq(_Constant().eval(), _const_path, _probe), 1.0)
        # a perfect 1.0 that proves nothing must never pass silently
        assert any('vacuous' in str(w.message) for w in _caught), [str(w.message) for w in _caught]

        _varied_path = export_onnx(_Passthrough().eval(), _probe, Path(_tmp)/'varied.onnx',
                                   dynamic_batch=False, optimize=False)
        with warnings.catch_warnings(record=True) as _caught:
            warnings.simplefilter('always')
            test_eq(verify_qdq(_Passthrough().eval(), _varied_path, _probe), 1.0)
        assert len(set(_probe.argmax(-1).tolist())) > 1, "this probe must produce varied predictions"
        assert not any('vacuous' in str(w.message) for w in _caught), \
            "a check with varied reference predictions is not vacuous and must not say it is"

In [ ]:
# --- export_qdq refuses the precisions it cannot write, and says which one it was given ---
# (these live here rather than in the exporter notebook because the refusal is driven by the spec
#  `Quantizer` attaches to the model)
with tempfile.TemporaryDirectory() as _tmp:
    _out = Path(_tmp)/'refused.onnx'

    # INT4 weights: opset 18 has no INT4 Q/DQ pair. The refusal is spec-driven, so it can be pinned
    # without the INT4 kernels this environment may not have.
    _int4_model = nn.Linear(8, 8)
    setattr(_int4_model, SPEC_ATTR, _resolve_spec('torchao', weight_bits=4))
    with ExceptionExpected(ValueError, regex="W4A16"):
        export_qdq(_int4_model, torch.randn(1, 8), _out)
    assert not _out.exists(), "a refused export must not leave a file behind"

    # weight-only INT8: no activation Q/DQ pair to write at all
    if _HAS_TORCHAO:
        _wo = Quantizer(backend='torchao', method='int8_weight_only').quantize(
            nn.Sequential(nn.Linear(8, 8)).eval())
        test_eq(quant_spec(_wo).label, 'W8A16')
        with ExceptionExpected(ValueError, regex="W8A16"):
            export_qdq(_wo, torch.randn(1, 8), _out)
        # ...and a per-layer W8A16 model is refused for the very same reason: the refusal reads the
        # precision cell, which a layer list does not change
        if _HAS_FQN_CONFIG:
            _wo_mixed = Quantizer(backend='torchao', weight_bits={'0': 16, '1': 8}, act_bits=16).quantize(
                nn.Sequential(nn.Linear(8, 8), nn.Linear(8, 8)).eval())
            test_eq(quant_spec(_wo_mixed).layer_bits, {'0': 16, '1': 8})
            with ExceptionExpected(ValueError, regex="W8A16"):
                export_qdq(_wo_mixed, torch.randn(1, 8), _out)
        assert not _out.exists(), "a refused export must not leave a file behind"

    # a legacy FX model: quantized, but not as a portable Q/DQ graph
    _fx = Quantizer().quantize(_TinyConvNet().eval(), _calib)
    with ExceptionExpected(ValueError, regex="backend='x86'"):
        export_qdq(_fx, _sample, _out)

    # an untagged model is exported exactly as before: the guard only reads what Quantizer wrote
    test_eq(quant_spec(nn.Linear(4, 4)), None)
    if _HAS_PT2E and all(_has_package(p) for p in ('onnx', 'onnxscript', 'onnxruntime')):
        _stripped = Quantizer(backend='pt2e').quantize(_model, _calib)
        delattr(_stripped, SPEC_ATTR)
        assert export_qdq(_stripped, _sample, Path(_tmp)/'untagged.onnx').exists()

In [ ]:
#| slow
# Integration test: torchao INT8 on transformer
if _HAS_TORCHAO:
    _encoder = nn.TransformerEncoderLayer(d_model=64, nhead=4, dim_feedforward=128, batch_first=True)
    _transformer = nn.TransformerEncoder(_encoder, num_layers=2).eval()
    _x = torch.randn(2, 10, 64)

    _tq = Quantizer(backend='torchao', method='int8_weight_only').quantize(_transformer)
    _out = _tq(_x)
    test_eq(_out.shape, (2, 10, 64))
    assert torch.isfinite(_out).all(), "torchao INT8 transformer produced non-finite outputs"

# --- per-layer widths on the same transformer: half of its Linear layers left in floating point ---
if _HAS_TORCHAO and _HAS_FQN_CONFIG:
    def _fresh_encoder():
        "A deterministically initialized TransformerEncoder, so two quantizations are comparable"
        torch.manual_seed(0)
        _layer = nn.TransformerEncoderLayer(d_model=64, nhead=4, dim_feedforward=128, batch_first=True)
        return nn.TransformerEncoder(_layer, num_layers=2).eval()

    _te_names = Quantizer(backend='torchao', method='int8_weight_only')._quantizable_names(_fresh_encoder())
    # the feed-forward Linears, and NOT MultiheadAttention's out_proj, which torchao's filter skips
    test_eq(_te_names, ['layers.0.linear1', 'layers.0.linear2', 'layers.1.linear1', 'layers.1.linear2'])

    # GOLDEN on a real model: naming every Linear 8 reproduces the uniform artifact byte for byte
    _te_uniform = Quantizer(backend='torchao', method='int8_weight_only').quantize(_fresh_encoder())
    _te_all8 = Quantizer(backend='torchao', weight_bits={n: 8 for n in _te_names},
                         act_bits=16).quantize(_fresh_encoder())
    test_eq(_state_digest(_te_all8), _state_digest(_te_uniform))

    # half of them at 16: the artifact really is mixed, and it still runs
    _half = {'layers.0.linear1': 16, 'layers.1.linear1': 16}
    _te_mixed = Quantizer(backend='torchao', weight_bits=_half, act_bits=16).quantize(_fresh_encoder())
    for _n in _te_names:
        test_eq(_weight_root(_te_mixed, _n), 'torch' if _n in _half else 'torchao')
    # out_proj stays float on BOTH paths, and `in_proj_weight` is a bare Parameter, never a named layer
    for _m in (_te_uniform, _te_mixed):
        test_eq(_weight_root(_m, 'layers.0.self_attn.out_proj'), 'torch')
        test_eq(type(_m.layers[0].self_attn.in_proj_weight).__module__.split('.')[0], 'torch')
    _te_out = _te_mixed(torch.randn(2, 10, 64))
    test_eq(_te_out.shape, (2, 10, 64))
    assert torch.isfinite(_te_out).all(), "a per-layer torchao transformer produced non-finite outputs"
    test_eq(quant_spec(_te_mixed).layer_bits, _half)
    # weight-only quantization is a SIZE lever: leaving two layers float costs bytes, and that is the
    # whole point of being able to name them
    assert len(_state_bytes(_te_uniform)) < len(_state_bytes(_te_mixed)) < len(_state_bytes(_fresh_encoder()))

    # ViT-Tiny is the shape this feature is for: a Linear-heavy model where a handful of layers can be
    # held out of the quantization by name
    try:
        import timm
    except ImportError:
        timm = None
    if timm is not None:
        _vit = timm.create_model('vit_tiny_patch16_224', pretrained=False, num_classes=10).eval()
        _vit_names = Quantizer(backend='torchao', method='int8_weight_only')._quantizable_names(_vit)
        assert len(_vit_names) > 20, _vit_names
        _vit_fp = {n: 16 for n in _vit_names[:4]}
        _vit_q = Quantizer(backend='torchao', weight_bits=_vit_fp, act_bits=16).quantize(_vit)
        for _n in _vit_names:
            test_eq(_weight_root(_vit_q, _n), 'torch' if _n in _vit_fp else 'torchao')
        _vit_out = _vit_q(torch.randn(1, 3, 224, 224))
        test_eq(_vit_out.shape, (1, 10))
        assert torch.isfinite(_vit_out).all(), "a per-layer torchao ViT produced non-finite outputs"
        test_eq(quant_spec(_vit_q).layer_bits, _vit_fp)

In [ ]:
#| slow
# Integration test: pt2e INT8 on a real ResNet-18, exported to a QDQ ONNX graph
if _HAS_PT2E:
    from torchvision.models import resnet18

    torch.manual_seed(0)
    _rn = resnet18(weights=None).eval()
    _rn_sample = torch.randn(2, 3, 64, 64)
    _rn_calib = [(torch.randn(2, 3, 64, 64), torch.randint(0, 1000, (2,))) for _ in range(3)]

    _rn_q = Quantizer(backend='pt2e').quantize(_rn, _rn_calib, max_calibration_samples=6)
    _rn_out = _rn_q(_rn_sample)
    test_eq(_rn_out.shape, (2, 1000))
    assert torch.isfinite(_rn_out).all()
    assert all(int(b.abs().max()) == 0 for name, b in _rn_q.named_buffers() if 'zero_point' in name)

    if all(_has_package(p) for p in ('onnx', 'onnxscript', 'onnxruntime')):
        with tempfile.TemporaryDirectory() as _tmp:
            _rn_path = export_qdq(_rn_q, _rn_sample, Path(_tmp)/'resnet18_qdq.onnx')
            _rn_stats = qdq_stats(_rn_path)
            # one per-channel weight tensor per conv (20) plus the classifier
            test_eq(_rn_stats.n_per_channel, 21)
            assert _rn_stats.n_quantize > 20, _rn_stats
            test_eq(_rn_stats.n_nonzero_zero_point, 0)

            # Logit parity on the SAME batch. Measured here (torch 2.9.1, onnxruntime CPU):
            # max|Δlogit| = 6.6e-2 same-batch (ORT runs real INT8 kernels, so it is not bit-exact)
            # against 5.3e-1 when the ONNX arm is fed a different batch. _RN_ATOL sits between them.
            # No argmax agreement is read here: this ResNet-18 is untrained and answers the same class
            # to every input, so the agreement would be 1.0 against any graph at all — and giving it a
            # spread head would move the very tolerances this cell pins. The default suite carries the
            # argmax check, on a model whose predictions vary.
            _RN_ATOL = 0.2
            _rn_session = ONNXModel(_rn_path)
            with torch.no_grad(): _rn_pt = _rn_q(_rn_sample).numpy()
            _rn_onnx = _rn_session(_rn_sample).numpy()
            assert np.allclose(_rn_pt, _rn_onnx, atol=_RN_ATOL), \
                f"max|Δlogit| = {np.abs(_rn_pt - _rn_onnx).max():.3e}"
            assert not np.allclose(_rn_pt, _rn_session(torch.randn(2, 3, 64, 64)).numpy(), atol=_RN_ATOL), \
                "the parity check cannot tell one input batch from another — it proves nothing"

            # the same model asked for opset 17: rewritten, checked and run — the same graph, node for
            # node, with the ReduceMean axes written where opset 17 reads them
            _rn17 = export_qdq(_rn_q, _rn_sample, Path(_tmp)/'resnet18_qdq17.onnx', opset_version=17)
            _rn_m17, _rn_m18 = onnx.load(str(_rn17)), onnx.load(str(_rn_path))
            test_eq([_o.version for _o in _rn_m17.opset_import if _o.domain in ('', 'ai.onnx')], [17])
            _rn_reduce = [_n for _n in _rn_m17.graph.node if _n.op_type == 'ReduceMean']
            assert _rn_reduce, "this post-condition needs the pooling this model has"
            assert all(len(_n.input) == 1 and any(_a.name == 'axes' for _a in _n.attribute)
                       for _n in _rn_reduce), _rn_reduce
            test_eq([_n.op_type for _n in _rn_m17.graph.node], [_n.op_type for _n in _rn_m18.graph.node])
            test_eq(qdq_stats(_rn17).as_dict(), _rn_stats.as_dict())
            assert ONNXModel(_rn17)(_rn_sample).numpy().tobytes() == _rn_onnx.tobytes(), \
                "opset 17 and opset 18 must answer with the same logits, bit for bit"

In [ ]:
#| slow
# Integration test: a real ResNet-18 quantized through the precision grammar, then exported
if _HAS_PT2E:
    from torchvision.models import resnet18

    torch.manual_seed(0)
    _g_rn = resnet18(weights=None).eval()
    _g_sample = torch.randn(2, 3, 64, 64)
    _g_calib = [(torch.randn(2, 3, 64, 64), torch.randint(0, 1000, (2,))) for _ in range(3)]

    _g_quantizer = Quantizer(backend='pt2e', weight_bits=8, act_bits=8, qscheme='per_channel',
                             symmetric=True)
    _g_q = _g_quantizer.quantize(_g_rn, _g_calib, max_calibration_samples=6)
    test_eq(_g_q(_g_sample).shape, (2, 1000))
    test_eq(quant_spec(_g_q).label, 'W8A8')
    test_eq(quant_spec(_g_q).exports, True)
    # naming the default precision must not change the artifact
    assert _same_state(_g_q, Quantizer(backend='pt2e').quantize(_g_rn, _g_calib, max_calibration_samples=6))

    if all(_has_package(p) for p in ('onnx', 'onnxscript', 'onnxruntime')):
        with tempfile.TemporaryDirectory() as _tmp:
            _g_path = export_qdq(_g_q, _g_sample, Path(_tmp)/'resnet18_grammar.onnx')
            _g_stats = qdq_stats(_g_path)
            # the symmetry the spec promised is the symmetry the file carries
            test_eq(_g_stats.n_nonzero_zero_point, 0)
            test_eq(_g_stats.n_per_channel, 21)   # one per-channel weight per conv (20) plus the classifier

            # same-batch logit parity, with the same calibrated tolerance as the default-config run
            _G_ATOL = 0.2
            _g_session = ONNXModel(_g_path)
            with torch.no_grad(): _g_pt = _g_q(_g_sample).numpy()
            assert np.allclose(_g_pt, _g_session(_g_sample).numpy(), atol=_G_ATOL)
            assert not np.allclose(_g_pt, _g_session(torch.randn(2, 3, 64, 64)).numpy(), atol=_G_ATOL), \
                "the parity check cannot tell one input batch from another — it proves nothing"

            # per-tensor weights: same model, one scale per weight tensor, still exportable
            _g_pt_model = Quantizer(backend='pt2e', qscheme='per_tensor').quantize(
                _g_rn, _g_calib, max_calibration_samples=6)
            _g_pt_stats = qdq_stats(export_qdq(_g_pt_model, _g_sample, Path(_tmp)/'resnet18_per_tensor.onnx'))
            test_eq(_g_pt_stats.n_per_channel, 0)
            test_eq(_g_pt_stats.n_nonzero_zero_point, 0)

In [ ]:
#| slow
# Integration test: a real ResNet-18 through both Q/DQ placements, exported and read back
if _HAS_PT2E:
    from torchvision.models import resnet18

    torch.manual_seed(0)
    _pl_rn = resnet18(weights=None).eval()
    _pl_sample, _pl_other = torch.randn(2, 3, 64, 64), torch.randn(2, 3, 64, 64)
    _pl_calib = [(torch.randn(2, 3, 64, 64), torch.randint(0, 1000, (2,))) for _ in range(3)]

    _pl_models = {_name: Quantizer(backend='pt2e', qdq_placement=_name).quantize(
                      _pl_rn, _pl_calib, max_calibration_samples=6)
                  for _name in ('per_op', 'skip_conv_add')}
    for _arm, _arm_model in _pl_models.items():
        test_eq((_arm, quant_spec(_arm_model).qdq_placement), (_arm, _arm))
        assert torch.isfinite(_arm_model(_pl_sample)).all(), _arm
        # the placement moves pairs; it does not make the remaining ones affine
        assert all(int(b.abs().max()) == 0
                   for n, b in _arm_model.named_buffers() if 'zero_point' in n)

    # the two arms really compute different things — a pass that quietly did nothing cannot pass here
    with torch.no_grad():
        _pl_delta = float((_pl_models['per_op'](_pl_sample) -
                           _pl_models['skip_conv_add'](_pl_sample)).abs().max())
    assert _pl_delta > 0, "the two placements produced the same arithmetic on ResNet-18"

    # ONE VARIABLE, on a real model: every pair the skip arm keeps carries the per_op arm's scale,
    # and the 16 it does not keep are the quantize and the dequantize of the eight cleared edges
    _pl_params = {_arm: _qdq_params(_arm_model) for _arm, _arm_model in _pl_models.items()}
    test_eq([k for k in set(_pl_params['per_op']) & set(_pl_params['skip_conv_add'])
             if _pl_params['per_op'][k] != _pl_params['skip_conv_add'][k]], [])
    test_eq(len(set(_pl_params['skip_conv_add']) - set(_pl_params['per_op'])), 0)
    test_eq(len(set(_pl_params['per_op']) - set(_pl_params['skip_conv_add'])), 16)

    if all(_has_package(p) for p in ('onnx', 'onnxscript', 'onnxruntime')):
        with tempfile.TemporaryDirectory() as _tmp:
            _tmp = Path(_tmp)
            _pl_paths = {_arm: export_qdq(_arm_model, _pl_sample, _tmp/f'resnet18_{_arm}.onnx')
                         for _arm, _arm_model in _pl_models.items()}
            _pl_stats = {_arm: qdq_stats(_path) for _arm, _path in _pl_paths.items()}

            # ResNet-18 has eight residual additions, and this placement clears every one of them
            test_eq(_pl_stats['per_op'].n_unquantized_conv_add, 0)
            test_eq(_pl_stats['skip_conv_add'].n_unquantized_conv_add, 8)
            test_eq(_pl_stats['per_op'].n_quantize - _pl_stats['skip_conv_add'].n_quantize, 8)
            test_eq(_pl_stats['per_op'].n_dequantize - _pl_stats['skip_conv_add'].n_dequantize, 8)
            for _arm, _arm_stats in _pl_stats.items():
                test_eq((_arm, _arm_stats.n_nonzero_zero_point), (_arm, 0))
                test_eq((_arm, _arm_stats.n_per_channel), (_arm, 21))  # one per conv (20) plus the fc

            # Parity is checked PER ARM, on a LOGIT bound: the arms differ on purpose, so comparing
            # one arm's file to the other arm's model would only measure the option — and argmax
            # agreement is vacuous on an untrained network (verify_qdq says so itself).
            # Measured here (torch 2.9.1, onnxruntime CPU): 6.9e-2 same-batch on both arms, against
            # 5.3e-1 when the ONNX arm is fed a different batch. _PL_ATOL sits between the two.
            _PL_ATOL = 0.2
            for _arm, _arm_model in _pl_models.items():
                _arm_session = ONNXModel(_pl_paths[_arm])
                with torch.no_grad(): _pl_pt = _arm_model(_pl_sample).numpy()
                _pl_onnx = _arm_session(_pl_sample).numpy()
                assert np.allclose(_pl_pt, _pl_onnx, atol=_PL_ATOL), \
                    f"{_arm}: max|Δlogit| = {np.abs(_pl_pt - _pl_onnx).max():.3e}"
                assert not np.allclose(_pl_pt, _arm_session(_pl_other).numpy(), atol=_PL_ATOL), \
                    f"{_arm}: the parity check cannot tell one input batch from another"

In [ ]:
#| slow
# Integration test: the pt2e default on a TRAINED EfficientNet-B0, scored against its own FP32 parent
# on real, class-balanced images. Every other pt2e test in this notebook runs on an untrained network,
# whose predictions carry no signal to lose — this is the cell where an activation grid too coarse to
# keep a model's answers shows up at all. (Imagenette-160 and the torchvision weights are downloaded
# once and cached.)
if _HAS_PT2E:
    from fastai.data.external import untar_data, URLs
    from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
    from PIL import Image

    # The ten Imagenette classes, as indices into the 1000-way ImageNet head, checked against
    # torchvision's own label list: a wrong index would depress the FP32 score and leave this cell
    # measuring the mapping instead of the quantization.
    _EB0_CLASSES = {'n01440764': (0, 'tench'),             'n02102040': (217, 'English springer'),
                    'n02979186': (482, 'cassette player'), 'n03000684': (491, 'chain saw'),
                    'n03028079': (497, 'church'),          'n03394916': (566, 'French horn'),
                    'n03417042': (569, 'garbage truck'),   'n03425413': (571, 'gas pump'),
                    'n03445777': (574, 'golf ball'),       'n03888257': (701, 'parachute')}
    _EB0_WEIGHTS = EfficientNet_B0_Weights.IMAGENET1K_V1
    for _idx, _name in _EB0_CLASSES.values():
        test_eq(_EB0_WEIGHTS.meta['categories'][_idx], _name)

    _eb0_root = untar_data(URLs.IMAGENETTE_160)
    _eb0_tfm = _EB0_WEIGHTS.transforms()  # resize 256 -> centre crop 224 -> ImageNet normalization
    _EB0_BS = 8                           # torch.export freezes the batch size: this one calibrates,
                                          # scores and exports, and no other batch size runs the graph
    _EB0_N_CALIB, _EB0_N_VAL = 8 * _EB0_BS, 25 * _EB0_BS

    def _eb0_files(split, per_class):
        "The first `per_class` images of every class, interleaved so that any prefix stays balanced"
        _groups = [sorted((_eb0_root/split/_w).glob('*.JPEG'))[:per_class] for _w in sorted(_EB0_CLASSES)]
        return [_f for _row in zip(*_groups) for _f in _row]

    def _eb0_batches(files):
        "`files` decoded and preprocessed, in batches of `_EB0_BS`"
        return [torch.stack([_eb0_tfm(Image.open(_f).convert('RGB')) for _f in files[_i:_i + _EB0_BS]])
                for _i in range(0, len(files), _EB0_BS)]

    def _eb0_preds(model, batches):
        "The class each batch row is given, over every batch"
        with torch.no_grad(): return torch.cat([model(_b).argmax(-1) for _b in batches])

    _eb0_calib_files = _eb0_files('train', 7)[:_EB0_N_CALIB]  # 7 per class covers 64 after interleaving
    _eb0_val_files = _eb0_files('val', _EB0_N_VAL // len(_EB0_CLASSES))
    test_eq((len(_eb0_calib_files), len(_eb0_val_files)), (_EB0_N_CALIB, _EB0_N_VAL))
    _eb0_calib = _eb0_batches(_eb0_calib_files)  # bare batches: calibration reads the input, not a label
    _eb0_val = _eb0_batches(_eb0_val_files)
    _eb0_labels = torch.tensor([_EB0_CLASSES[_f.parent.name][0] for _f in _eb0_val_files])

    # The reference comes first, and so does the guard on it: a quantized model can only be compared
    # against a parent that classifies, and a broken mapping or preprocessing has to fail here — in
    # seconds — rather than after the quantization.
    _eb0_fp32 = efficientnet_b0(weights=_EB0_WEIGHTS).eval()
    _eb0_ref = _eb0_preds(_eb0_fp32, _eb0_val)
    _eb0_fp32_top1 = float((_eb0_ref == _eb0_labels).float().mean())
    assert _eb0_fp32_top1 >= 0.6, \
        f"FP32 top-1 is {_eb0_fp32_top1:.2%}: what is wrong here is the preprocessing, not the quantization"
    # ...and what a model answering ONE class to everything would score against that reference. The
    # threshold has to clear this floor, or agreeing with the parent would prove nothing — the same
    # trap `verify_qdq` refuses by name.
    _eb0_floor = float(torch.bincount(_eb0_ref).max()) / len(_eb0_ref)

    # Measured here (torch 2.9.1, CPU, ONE fixed calibration draw of 64 images, 200 class-balanced
    # validation images): argmax agreement with the FP32 parent is 0.5% when the activations are
    # observed by a non-clipping min/max observer, against 34.5% with the clipping histogram observer
    # this config asks for. _EB0_MIN_AGREEMENT sits between the two, 9.5 points under the measured one.
    # This does not restore the model: on the same set the quantized top-1 is 32.00% against the FP32
    # parent's 73.00%. The residual cost is the symmetric activation grid itself; `symmetric=False`
    # trades it for non-zero zero-points.
    _EB0_MIN_AGREEMENT = 0.25
    assert _eb0_floor < _EB0_MIN_AGREEMENT, \
        f"a constant predictor already agrees {_eb0_floor:.2%} of the time — this eval set proves nothing"

    _eb0_q = Quantizer(backend='pt2e').quantize(_eb0_fp32, _eb0_calib,
                                                max_calibration_samples=_EB0_N_CALIB)  # all of them
    # the portability guarantee this config exists for, on a trained model
    assert all(int(_b.abs().max()) == 0 for _n, _b in _eb0_q.named_buffers() if 'zero_point' in _n)

    _eb0_q_pred = _eb0_preds(_eb0_q, _eb0_val)
    _eb0_agree = float((_eb0_q_pred == _eb0_ref).float().mean())
    assert _eb0_agree >= _EB0_MIN_AGREEMENT, \
        f"INT8 agrees with its own FP32 parent on {_eb0_agree:.2%} of {len(_eb0_ref)} images " \
        f"(INT8 top-1 {float((_eb0_q_pred == _eb0_labels).float().mean()):.2%}, FP32 top-1 " \
        f"{_eb0_fp32_top1:.2%}, constant-predictor floor {_eb0_floor:.2%})"

    # ...and the same guarantee read off the FILE rather than off the model's buffers. The captured
    # graph is frozen at the calibration batch size, so it is exported with a sample of that shape.
    if all(_has_package(_p) for _p in ('onnx', 'onnxscript', 'onnxruntime')):
        with tempfile.TemporaryDirectory() as _tmp:
            _eb0_path = export_qdq(_eb0_q, _eb0_val[0], Path(_tmp)/'efficientnet_b0_qdq.onnx')
            _eb0_stats = qdq_stats(_eb0_path)
            test_eq(_eb0_stats.n_nonzero_zero_point, 0)
            # ...on a file that really carries the pairs — a graph with none would pass that for free
            test_eq(_eb0_stats.n_per_channel, 82)  # one per-channel weight per conv (81) plus the classifier
            assert _eb0_stats.n_quantize > 100, _eb0_stats